In [ ]:
# ================================================================
# RANDOM 10% PARAMETER TRAINING WITH DUAL-ACCURACY WEIGHT UPDATE
# ================================================================
#
# Architecture: ResNet-18
# Training: CIFAR-100
# Validation: CIFAR-100 + MiniImageNet
# Parameter selection: Random 10% from every eligible tensor
# Iterations: 40
# Epochs per iteration: 5
#
# Weight update:
#
# F = lambda * delta_cifar + (1-lambda) * delta_mini
#
# beta = sigmoid(k * F)
#
# W_next = (1-beta)*W0 + beta*W1
#
# ================================================================

import os
import json
import time
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from torch.utils.data import DataLoader, random_split
from tqdm.auto import tqdm


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("DEVICE:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# Dataset paths - CHANGE ACCORDING TO YOUR KAGGLE DATASETS

CIFAR_PATH = "/kaggle/working/cifar100"

MINI_PATH = "/kaggle/input/datasets/deeptrial/miniimagenet"

IMAGENET_MAPPING_PATH = (
    "/kaggle/input/imagenet-class-index/imagenet_class_index.json"
)

OUTPUT_DIR = "/kaggle/working/Random_10_Percent_ResNet18"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# Experiment settings

NUM_ITERATIONS = 40

EPOCHS_PER_ITERATION = 5

BATCH_SIZE = 128

NUM_WORKERS = 2

SELECT_RATIO = 0.10

LEARNING_RATE = 0.001

LAMBDA = 0.5

K = 0.1


# ================================================================
# 2. DATA TRANSFORMS
# ================================================================

train_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )

])


test_transform = transforms.Compose([

    transforms.Resize((224, 224)),

    transforms.ToTensor(),

    transforms.Normalize(
        [0.485, 0.456, 0.406],
        [0.229, 0.224, 0.225]
    )

])



In [ ]:
# ================================================================
# 3. CIFAR-100 DATASET
# ================================================================

cifar_full = datasets.CIFAR100(
    root=CIFAR_PATH,
    train=True,
    download=True,
    transform=train_transform
)

train_size = int(0.9 * len(cifar_full))

val_size = len(cifar_full) - train_size

generator = torch.Generator().manual_seed(SEED)

train_subset, val_subset = random_split(
    cifar_full,
    [train_size, val_size],
    generator=generator
)

cifar_val_full = datasets.CIFAR100(
    root=CIFAR_PATH,
    train=True,
    download=True,
    transform=test_transform
)

val_subset.dataset = cifar_val_full


cifar_train_loader = DataLoader(
    train_subset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

cifar_val_loader = DataLoader(
    val_subset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

print("\nCIFAR-100 TRAIN:", len(train_subset))
print("CIFAR-100 VALIDATION:", len(val_subset))




In [ ]:
# ================================================================
# 4. MINI-IMAGENET DATASET
# ================================================================

MINI_PATH = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini"

MINI_IMAGES_PATH = os.path.join(MINI_PATH, "images")

IMAGENET_MAPPING_PATH = os.path.join(
    MINI_PATH, "imagenet_class_index.json"
)

# ------------------------------------------------
# LOAD IMAGENET CLASS MAPPING
# ------------------------------------------------

with open(IMAGENET_MAPPING_PATH, "r") as f:
    imagenet_mapping = json.load(f)

# WNID -> Original ImageNet class index
wnid_to_idx = {
    value[0]: int(key)
    for key, value in imagenet_mapping.items()
}

# ------------------------------------------------
# LOAD MINI-IMAGENET DATASET
# ------------------------------------------------

mini_val_dataset = datasets.ImageFolder(
    root=MINI_IMAGES_PATH,
    transform=test_transform
)

# ------------------------------------------------
# MAP MINI-IMAGENET CLASSES TO IMAGENET INDICES
# ------------------------------------------------

mini_class_mapping = {}

for class_name, class_idx in mini_val_dataset.class_to_idx.items():

    if class_name not in wnid_to_idx:
        raise ValueError(
            f"Class {class_name} missing from ImageNet mapping."
        )

    mini_class_mapping[class_idx] = wnid_to_idx[class_name]


# Update targets to original ImageNet class indices
mini_val_dataset.samples = [
    (path, mini_class_mapping[target])
    for path, target in mini_val_dataset.samples
]

mini_val_dataset.targets = [
    target for _, target in mini_val_dataset.samples
]

# ------------------------------------------------
# DATALOADER
# ------------------------------------------------

mini_val_loader = DataLoader(
    mini_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

# ------------------------------------------------
# INFORMATION
# ------------------------------------------------

print("\n========== MINI-IMAGENET VALIDATION ==========")

print("Dataset path:", MINI_IMAGES_PATH)
print("Total images:", len(mini_val_dataset))
print("Total classes:", len(mini_class_mapping))
print("ImageNet mapping classes:", len(wnid_to_idx))

print("==============================================")

In [ ]:

# ================================================================
# 5. RESNET-18 WITH TWO CLASSIFICATION HEADS
# ================================================================

class DualHeadResNet18(nn.Module):

    def __init__(self):

        super().__init__()

        base = models.resnet18(
            weights=ResNet18_Weights.IMAGENET1K_V1
        )

        self.backbone = nn.Sequential(
            *list(base.children())[:-1]
        )

        self.imagenet_head = base.fc

        self.cifar_head = nn.Linear(512, 100)


    def forward(self, x):

        features = self.backbone(x)

        features = torch.flatten(features, 1)

        cifar_output = self.cifar_head(features)

        imagenet_output = self.imagenet_head(features)

        return cifar_output, imagenet_output


model = DualHeadResNet18().to(device)

print("\nMODEL CREATED")

print("Total parameters:",
      sum(p.numel() for p in model.parameters()))



In [ ]:
# ================================================================
# 6. RANDOM 10% PARAMETER SELECTION
# ================================================================

def generate_masks(model):

    masks = {}

    total = 0

    selected = 0

    for name, param in model.named_parameters():

        # ImageNet classification head is completely frozen.
        if name.startswith("imagenet_head"):
            continue

        n = param.numel()

        k = max(1, int(n * SELECT_RATIO))

        mask = torch.zeros(
            n,
            dtype=torch.bool,
            device=param.device
        )

        indices = torch.randperm(
            n,
            device=param.device
        )[:k]

        mask[indices] = True

        masks[name] = mask.reshape(param.shape)

        total += n

        selected += k

    print(
        f"\nSelected {selected:,}/{total:,} "
        f"parameters = {100*selected/total:.2f}%"
    )

    return masks


# ================================================================
# 7. SAVE MODEL PARAMETERS
# ================================================================

def save_weights(model):

    return {
        name: param.detach().clone()
        for name, param in model.named_parameters()
    }


def load_weights(model, weights):

    with torch.no_grad():

        for name, param in model.named_parameters():

            param.copy_(weights[name])



In [ ]:


# ================================================================
# 8. WEIGHT INTERPOLATION
# ================================================================

def interpolate_weights(model, W0, W1, beta):

    with torch.no_grad():

        for name, param in model.named_parameters():

            param.copy_(
                (1-beta)*W0[name] + beta*W1[name]
            )



In [ ]:

# ================================================================
# 9. EVALUATION
# ================================================================

@torch.no_grad()
def evaluate(model, loader, dataset_type):

    model.eval()

    correct = 0

    total = 0

    for images, labels in loader:

        images = images.to(
            device,
            non_blocking=True
        )

        labels = labels.to(
            device,
            non_blocking=True
        )

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16,
            enabled=(device.type == "cuda")
        ):

            cifar_output, mini_output = model(images)

        if dataset_type == "cifar":

            predictions = cifar_output.argmax(1)

        else:

            predictions = mini_output.argmax(1)

        correct += (predictions == labels).sum().item()

        total += labels.size(0)

    return 100 * correct / total



In [ ]:
# ================================================================
# 10. TRAINING ONE ITERATION
# ================================================================

def train_iteration(model, masks):

    criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        [
            param
            for name, param in model.named_parameters()
            if name in masks
        ],
        lr=LEARNING_RATE,
        momentum=0.9,
        weight_decay=0.0
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(device.type == "cuda")
    )

    for epoch in range(EPOCHS_PER_ITERATION):

        model.train()

        # Keep BatchNorm running statistics fixed.
        for module in model.modules():

            if isinstance(module, nn.BatchNorm2d):
                module.eval()

        correct = 0

        total = 0

        loss_sum = 0

        start = time.time()

        progress = tqdm(
            cifar_train_loader,
            desc=f"Epoch {epoch+1}/{EPOCHS_PER_ITERATION}"
        )

        for images, labels in progress:

            images = images.to(
                device,
                non_blocking=True
            )

            labels = labels.to(
                device,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16,
                enabled=(device.type == "cuda")
            ):

                cifar_output, _ = model(images)

                loss = criterion(cifar_output, labels)

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            # Apply random masks
            for name, param in model.named_parameters():

                if name in masks:

                    if param.grad is not None:

                        param.grad.mul_(masks[name])

            scaler.step(optimizer)

            scaler.update()

            loss_sum += loss.item()

            predictions = cifar_output.argmax(1)

            correct += (predictions == labels).sum().item()

            total += labels.size(0)

            progress.set_postfix(
                loss=f"{loss.item():.4f}",
                accuracy=f"{100*correct/total:.2f}%"
            )

        elapsed = time.time() - start

        print(
            f"Epoch {epoch+1}: "
            f"Loss={loss_sum/len(cifar_train_loader):.4f} | "
            f"Accuracy={100*correct/total:.2f}% | "
            f"Time={elapsed:.2f}s"
        )


In [ ]:



# ================================================================
# 11. MAIN ITERATIVE TRAINING
# ================================================================

results = []

W_current = save_weights(model)

print("\nSTARTING EXPERIMENT")

for iteration in range(NUM_ITERATIONS):

    print("\n" + "="*75)

    print(f"ITERATION {iteration+1}/{NUM_ITERATIONS}")

    print("="*75)

    iteration_start = time.time()

    # Restore previous iteration's final interpolated weights.
    load_weights(model, W_current)

    # W0: Beginning weights
    W0 = save_weights(model)

    # Accuracy before training
    cifar_before = evaluate(
        model,
        cifar_val_loader,
        "cifar"
    )

    mini_before = evaluate(
        model,
        mini_val_loader,
        "mini"
    )

    print("\nBEFORE TRAINING")

    print(f"CIFAR-100: {cifar_before:.3f}%")

    print(f"MiniImageNet: {mini_before:.3f}%")

    # Generate new random 10% selection for every iteration.
    masks = generate_masks(model)

    # Train ONLY on CIFAR-100.
    train_iteration(model, masks)

    # W1: Ending weights
    W1 = save_weights(model)

    # Accuracy after training
    cifar_after = evaluate(
        model,
        cifar_val_loader,
        "cifar"
    )

    mini_after = evaluate(
        model,
        mini_val_loader,
        "mini"
    )

    # Accuracy changes
    delta_cifar = cifar_after - cifar_before

    delta_mini = mini_after - mini_before

    # Dual-accuracy fitness
    fitness = (
        LAMBDA * delta_cifar
        + (1-LAMBDA) * delta_mini
    )

    # Sigmoid coefficient
    beta = 1 / (1 + np.exp(-np.clip(K*fitness, -60, 60)))

    # Calculate next iteration's weights.
    interpolate_weights(
        model,
        W0,
        W1,
        beta
    )

    W_current = save_weights(model)

    elapsed = time.time() - iteration_start

    # Record
    record = {

        "Iteration": iteration+1,

        "CIFAR_Before": cifar_before,

        "CIFAR_After": cifar_after,

        "Mini_Before": mini_before,

        "Mini_After": mini_after,

        "Delta_CIFAR": delta_cifar,

        "Delta_Mini": delta_mini,

        "Fitness": fitness,

        "Beta": beta,

        "Time_Minutes": elapsed/60

    }

    results.append(record)

    print("\nITERATION RESULTS")

    print(f"Delta CIFAR: {delta_cifar:+.4f}%")

    print(f"Delta Mini: {delta_mini:+.4f}%")

    print(f"Fitness: {fitness:+.5f}")

    print(f"Beta: {beta:.5f}")

    print(f"W0 contribution: {(1-beta)*100:.2f}%")

    print(f"W1 contribution: {beta*100:.2f}%")

    print(f"Time: {elapsed/60:.2f} minutes")

    # Save checkpoint
    torch.save(
        {
            "iteration": iteration+1,
            "model_state_dict": model.state_dict(),
            "W_current": W_current,
            "results": results
        },
        os.path.join(
            OUTPUT_DIR,
            f"iteration_{iteration+1}.pth"
        )
    )

    # Save results after every iteration
    pd.DataFrame(results).to_csv(
        os.path.join(OUTPUT_DIR, "results.csv"),
        index=False
    )



In [ ]:

# ================================================================
# 12. FINAL RESULTS
# ================================================================

print("\n" + "="*75)

print("EXPERIMENT COMPLETED")

print("="*75)

final_cifar = evaluate(
    model,
    cifar_val_loader,
    "cifar"
)

final_mini = evaluate(
    model,
    mini_val_loader,
    "mini"
)

print(f"\nFinal CIFAR-100 accuracy: {final_cifar:.3f}%")

print(f"Final MiniImageNet accuracy: {final_mini:.3f}%")

df = pd.DataFrame(results)

display(df)

df.to_csv(
    os.path.join(OUTPUT_DIR, "final_results.csv"),
    index=False
)

torch.save(
    model.state_dict(),
    os.path.join(OUTPUT_DIR, "final_model.pth")
)

print("\nAll results saved to:", OUTPUT_DIR)

## nxt


In [ ]:
# ================================================================
# DLA: RANDOM 10% PARAMETER ADAPTATION WITH TWO-DATASET TRAINING
# ResNet-18 | CIFAR-100 Cross-Entropy | MiniImageNet KL Distillation
# ================================================================
#
# Preserved from the supplied implementation:
#   - ResNet-18 initialized with ImageNet-1K pretrained weights
#   - New random 10% mask for EVERY eligible parameter tensor
#     at EVERY DLA iteration
#   - 5 epochs per iteration, maximum 40 iterations
#   - W0/W1 interpolation with sigmoid-adaptive beta
#   - CIFAR-100 accuracy is the ONLY DLA update/early-stop criterion
#
# Requested changes:
#   - CIFAR-100: CrossEntropyLoss
#   - MiniImageNet: teacher-student KL-divergence distillation
#   - MiniImageNet images are split class-wise into train/validation
#   - Strictly preserve unselected parameters during training AND
#     W0/W1 interpolation
#   - Stop after 3 consecutive iterations without CIFAR validation
#     accuracy improvement
#
# IMPORTANT:
#   MiniImageNet validation images are NEVER used for training.
#   MiniImageNet accuracy is reported, but does NOT control beta,
#   candidate acceptance, best-checkpoint selection, or patience.
#
# ================================================================

import os
import json
import time
import copy
import random
import glob
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights
from tqdm.auto import tqdm


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# CIFAR-100 will download here if it is not already present.
CIFAR_PATH = "/kaggle/working/cifar100"

# User-provided Kaggle dataset location.
MINI_ROOT = "/kaggle/input/datasets/deeptrial/miniimagenet"
MINI_PATH = os.path.join(MINI_ROOT, "ImageNet-Mini", "images")

# Keep the mapping path from the previously supplied script.
IMAGENET_MAPPING_PATH = (
    "/kaggle/input/imagenet-class-index/imagenet_class_index.json"
)

OUTPUT_DIR = "/kaggle/working/DLA_ResNet18_CIFAR100_MiniImageNet"
os.makedirs(OUTPUT_DIR, exist_ok=True)

NUM_ITERATIONS = 40
EPOCHS_PER_ITERATION = 5
PATIENCE = 3

BATCH_SIZE = 128
NUM_WORKERS = 2
SELECT_RATIO = 0.10

LEARNING_RATE = 0.001
MOMENTUM = 0.9

# MiniImageNet KL distillation settings.
DISTILLATION_TEMPERATURE = 2.0
KL_LOSS_WEIGHT = 1.0

# Original DLA sigmoid scaling parameter.
# beta = sigmoid(K * delta_CIFAR)
K = 0.1

# The original implementation uses beta = sigmoid(K * fitness).
# Fitness is now ONLY CIFAR-100 validation accuracy change.
# No MiniImageNet accuracy is used in this calculation.


def set_seed(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed()

print("DEVICE:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True


# ================================================================
# 2. PATH VALIDATION AND IMAGE TRANSFORMS
# ================================================================

if not os.path.isdir(MINI_PATH):
    raise FileNotFoundError(
        f"MiniImageNet image-folder path was not found:\n{MINI_PATH}\n"
        "Expected: .../ImageNet-Mini/images/<WNID>/<image files>"
    )

if not os.path.isfile(IMAGENET_MAPPING_PATH):
    # Try to locate the same mapping file somewhere under /kaggle/input.
    candidates = glob.glob(
        "/kaggle/input/**/imagenet_class_index.json",
        recursive=True
    )
    if candidates:
        IMAGENET_MAPPING_PATH = candidates[0]
        print("Found ImageNet mapping:", IMAGENET_MAPPING_PATH)
    else:
        raise FileNotFoundError(
            "imagenet_class_index.json was not found. Set "
            "IMAGENET_MAPPING_PATH to the correct Kaggle file."
        )

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

cifar_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

cifar_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

mini_train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])

mini_eval_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])


# ================================================================
# 3. CIFAR-100 TRAIN / VALIDATION SPLIT
# ================================================================

cifar_train_base = datasets.CIFAR100(
    root=CIFAR_PATH,
    train=True,
    download=True,
    transform=cifar_train_transform
)

# A separate dataset object ensures validation has deterministic transforms.
cifar_eval_base = datasets.CIFAR100(
    root=CIFAR_PATH,
    train=True,
    download=True,
    transform=cifar_eval_transform
)

num_cifar = len(cifar_train_base)
indices = np.arange(num_cifar)
rng = np.random.default_rng(SEED)
rng.shuffle(indices)

train_size = int(0.90 * num_cifar)
cifar_train_indices = indices[:train_size].tolist()
cifar_val_indices = indices[train_size:].tolist()

cifar_train_dataset = Subset(cifar_train_base, cifar_train_indices)
cifar_val_dataset = Subset(cifar_eval_base, cifar_val_indices)

cifar_train_loader = DataLoader(
    cifar_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

cifar_val_loader = DataLoader(
    cifar_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print("\nCIFAR-100 TRAIN:", len(cifar_train_dataset))
print("CIFAR-100 VALIDATION:", len(cifar_val_dataset))


# ================================================================
# 4. MINI-IMAGENET CLASS-WISE TRAIN / VALIDATION SPLIT
# ================================================================

with open(IMAGENET_MAPPING_PATH, "r") as f:
    imagenet_mapping = json.load(f)

# WNID -> original ImageNet-1K output index.
wnid_to_imagenet_idx = {
    value[0]: int(class_idx)
    for class_idx, value in imagenet_mapping.items()
}

# Build two ImageFolder instances so train and validation use different
# transforms, but use exactly the same sorted class-directory order.
mini_train_base = datasets.ImageFolder(
    root=MINI_PATH,
    transform=mini_train_transform
)
mini_eval_base = datasets.ImageFolder(
    root=MINI_PATH,
    transform=mini_eval_transform
)

if mini_train_base.classes != mini_eval_base.classes:
    raise RuntimeError("MiniImageNet train/eval class mappings differ.")

mini_classes = mini_train_base.classes
mini_class_to_idx = mini_train_base.class_to_idx
mini_idx_to_class = {v: k for k, v in mini_class_to_idx.items()}

missing_wnids = [
    wnid for wnid in mini_classes
    if wnid not in wnid_to_imagenet_idx
]
if missing_wnids:
    raise ValueError(
        "These MiniImageNet WNIDs are missing from "
        "imagenet_class_index.json:\n" + ", ".join(missing_wnids[:20])
    )

class_indices = defaultdict(list)
for sample_idx, (_, class_idx) in enumerate(mini_train_base.samples):
    class_indices[class_idx].append(sample_idx)

mini_train_indices = []
mini_val_indices = []

split_rng = np.random.default_rng(SEED)
for class_idx, class_sample_indices in class_indices.items():
    class_sample_indices = np.asarray(class_sample_indices)
    split_rng.shuffle(class_sample_indices)

    # 80% train / 20% validation independently within every class.
    class_train_count = max(1, int(0.80 * len(class_sample_indices)))
    if len(class_sample_indices) > 1:
        class_train_count = min(class_train_count, len(class_sample_indices) - 1)

    mini_train_indices.extend(class_sample_indices[:class_train_count].tolist())
    mini_val_indices.extend(class_sample_indices[class_train_count:].tolist())


class ImageNetMappedSubset(Dataset):
    """
    Reads an ImageFolder subset and maps its WNID class label to the
    corresponding 0..999 ImageNet-1K classifier index.
    """

    def __init__(self, imagefolder, indices, idx_to_class, wnid_mapping):
        self.imagefolder = imagefolder
        self.indices = list(indices)
        self.idx_to_class = idx_to_class
        self.wnid_mapping = wnid_mapping

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, item):
        original_index = self.indices[item]
        image, folder_class_idx = self.imagefolder[original_index]
        wnid = self.idx_to_class[folder_class_idx]
        imagenet_label = self.wnid_mapping[wnid]
        return image, imagenet_label


mini_train_dataset = ImageNetMappedSubset(
    mini_train_base,
    mini_train_indices,
    mini_idx_to_class,
    wnid_to_imagenet_idx
)

mini_val_dataset = ImageNetMappedSubset(
    mini_eval_base,
    mini_val_indices,
    mini_idx_to_class,
    wnid_to_imagenet_idx
)

mini_train_loader = DataLoader(
    mini_train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

mini_val_loader = DataLoader(
    mini_val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
    persistent_workers=(NUM_WORKERS > 0),
)

print("\nMINI-IMAGENET CLASSES:", len(mini_classes))
print("MINI-IMAGENET TRAIN:", len(mini_train_dataset))
print("MINI-IMAGENET VALIDATION:", len(mini_val_dataset))
print("MiniImageNet split: 80% train / 20% validation per class")


# ================================================================
# 5. STUDENT RESNET-18 WITH TWO HEADS
# ================================================================

class DualHeadResNet18(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()

        weights = ResNet18_Weights.IMAGENET1K_V1 if pretrained else None
        base = models.resnet18(weights=weights)

        self.backbone = nn.Sequential(*list(base.children())[:-1])

        # Original pretrained ImageNet-1K classifier (1000 outputs).
        self.imagenet_head = base.fc

        # New CIFAR-100 classifier (100 outputs).
        self.cifar_head = nn.Linear(512, 100)

    def forward(self, x):
        features = self.backbone(x)
        features = torch.flatten(features, 1)
        cifar_logits = self.cifar_head(features)
        imagenet_logits = self.imagenet_head(features)
        return cifar_logits, imagenet_logits


student = DualHeadResNet18(pretrained=True).to(DEVICE)

# Frozen teacher: exact ImageNet-pretrained ResNet-18.
# It provides soft targets for MiniImageNet KL distillation.
teacher = models.resnet18(
    weights=ResNet18_Weights.IMAGENET1K_V1
).to(DEVICE)
teacher.eval()
for parameter in teacher.parameters():
    parameter.requires_grad_(False)

print("\nSTUDENT MODEL CREATED")
print("Student parameters:", sum(p.numel() for p in student.parameters()))
print("Teacher is frozen and used only to generate MiniImageNet soft targets.")


# ================================================================
# 6. RANDOM 10% MASK FOR EACH ELIGIBLE PARAMETER TENSOR
# ================================================================

def generate_masks(model):
    """
    Generate a fresh independent random 10% mask for every trainable
    parameter tensor in the student at the start of each iteration.

    Each tensor gets exactly floor(10% * numel), with at least one
    selected element. The frozen teacher is not part of this model.
    """
    masks = {}
    total_parameters = 0
    selected_parameters = 0

    for name, parameter in model.named_parameters():
        if not parameter.requires_grad:
            continue

        number = parameter.numel()
        selected_count = max(1, int(number * SELECT_RATIO))

        flat_mask = torch.zeros(
            number,
            dtype=torch.bool,
            device=parameter.device
        )
        selected_indices = torch.randperm(
            number,
            device=parameter.device
        )[:selected_count]

        flat_mask[selected_indices] = True
        masks[name] = flat_mask.view_as(parameter)

        total_parameters += number
        selected_parameters += selected_count

        print(
            f"  {name:35s}: selected {selected_count:,}/{number:,} "
            f"({100.0 * selected_count / number:.2f}%)"
        )

    print(
        f"\nTOTAL SELECTED: {selected_parameters:,}/"
        f"{total_parameters:,} parameters "
        f"({100.0 * selected_parameters / total_parameters:.2f}%)"
    )
    return masks, total_parameters, selected_parameters


# ================================================================
# 7. WEIGHT SNAPSHOTS AND STRICT MASK ENFORCEMENT
# ================================================================

def save_weights(model):
    return {
        name: parameter.detach().clone()
        for name, parameter in model.named_parameters()
    }


def load_weights(model, weights):
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            parameter.copy_(weights[name])


def enforce_training_mask(model, masks, W0):
    """
    Restore every unselected parameter to its iteration-start value.
    Also mask momentum buffers so momentum cannot move unselected weights.
    """
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name not in masks:
                continue

            mask = masks[name]
            parameter.copy_(torch.where(mask, parameter, W0[name]))


def interpolate_weights_masked(model, W0, W1, masks, beta):
    """
    Apply the original DLA interpolation ONLY to selected coordinates.

    Selected:   (1-beta)*W0 + beta*W1
    Unselected: W0 exactly
    """
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            if name not in masks:
                # No mask means no parameter is eligible for change.
                parameter.copy_(W0[name])
                continue

            mask = masks[name]
            interpolated = (1.0 - beta) * W0[name] + beta * W1[name]
            parameter.copy_(torch.where(mask, interpolated, W0[name]))


def count_changed_parameters(model, W0):
    changed = 0
    total = 0
    with torch.no_grad():
        for name, parameter in model.named_parameters():
            total += parameter.numel()
            changed += torch.count_nonzero(parameter != W0[name]).item()
    return changed, total


# ================================================================
# 8. EVALUATION
# ================================================================

@torch.no_grad()
def evaluate_cifar(model, loader):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            cifar_logits, _ = model(images)

        predictions = cifar_logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / max(total, 1)


@torch.no_grad()
def evaluate_mini(model, loader):
    model.eval()
    correct = 0
    total = 0

    for images, labels in loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=(DEVICE.type == "cuda")
        ):
            _, imagenet_logits = model(images)

        predictions = imagenet_logits.argmax(dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    return 100.0 * correct / max(total, 1)


# ================================================================
# 9. TRAIN ONE DLA ITERATION
# ================================================================

def train_one_iteration(model, teacher_model, masks, W0, iteration_number):
    """
    Jointly train on:
      - CIFAR-100 batches with Cross-Entropy
      - MiniImageNet batches with teacher-student KL divergence

    Both losses train the same student ResNet-18. Only the randomly
    selected 10% parameter coordinates are allowed to change.
    """
    cifar_criterion = nn.CrossEntropyLoss()

    optimizer = optim.SGD(
        [p for p in model.parameters() if p.requires_grad],
        lr=LEARNING_RATE,
        momentum=MOMENTUM,
        weight_decay=0.0
    )

    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=(DEVICE.type == "cuda")
    )

    temperature = DISTILLATION_TEMPERATURE
    steps_per_epoch = max(len(cifar_train_loader), len(mini_train_loader))

    for epoch in range(EPOCHS_PER_ITERATION):
        model.train()
        teacher_model.eval()

        # Freeze BatchNorm running statistics, matching the original code.
        for module in model.modules():
            if isinstance(module, nn.BatchNorm2d):
                module.eval()

        cifar_iterator = iter(cifar_train_loader)
        mini_iterator = iter(mini_train_loader)

        cifar_loss_sum = 0.0
        mini_kl_sum = 0.0
        cifar_correct = 0
        cifar_total = 0
        mini_steps = 0

        start_time = time.time()
        progress = tqdm(
            range(steps_per_epoch),
            desc=f"Iteration {iteration_number} | Epoch "
                 f"{epoch + 1}/{EPOCHS_PER_ITERATION}"
        )

        for _ in progress:
            try:
                cifar_images, cifar_labels = next(cifar_iterator)
            except StopIteration:
                cifar_iterator = iter(cifar_train_loader)
                cifar_images, cifar_labels = next(cifar_iterator)

            try:
                mini_images, _mini_labels = next(mini_iterator)
            except StopIteration:
                mini_iterator = iter(mini_train_loader)
                mini_images, _mini_labels = next(mini_iterator)

            cifar_images = cifar_images.to(DEVICE, non_blocking=True)
            cifar_labels = cifar_labels.to(DEVICE, non_blocking=True)
            mini_images = mini_images.to(DEVICE, non_blocking=True)

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=(DEVICE.type == "cuda")
            ):
                # CIFAR-100 supervised classification.
                cifar_logits, _ = model(cifar_images)
                loss_cifar = cifar_criterion(cifar_logits, cifar_labels)

                # MiniImageNet teacher-student knowledge distillation.
                # Teacher outputs are fixed soft targets from pretrained
                # ImageNet-1K. MiniImageNet folder labels are mapped to
                # ImageNet-1K indices for evaluation, but KL uses the
                # complete 1000-class teacher/student distributions.
                with torch.no_grad():
                    teacher_logits = teacher_model(mini_images)

                _, student_mini_logits = model(mini_images)

                teacher_probabilities = F.softmax(
                    teacher_logits / temperature,
                    dim=1
                )
                student_log_probabilities = F.log_softmax(
                    student_mini_logits / temperature,
                    dim=1
                )

                loss_mini_kl = F.kl_div(
                    student_log_probabilities,
                    teacher_probabilities,
                    reduction="batchmean"
                ) * (temperature ** 2)

                total_loss = loss_cifar + KL_LOSS_WEIGHT * loss_mini_kl

            scaler.scale(total_loss).backward()
            scaler.unscale_(optimizer)

            # Apply the fresh random mask to every eligible gradient.
            for name, parameter in model.named_parameters():
                if name in masks and parameter.grad is not None:
                    parameter.grad.mul_(masks[name])

            scaler.step(optimizer)
            scaler.update()

            # Hard guarantee: restore all unselected weights after every
            # optimizer step. This also protects against optimizer-state
            # effects such as momentum.
            enforce_training_mask(model, masks, W0)

            # Explicitly mask momentum buffers as an additional safeguard.
            for name, parameter in model.named_parameters():
                if name in masks:
                    state = optimizer.state.get(parameter, {})
                    momentum_buffer = state.get("momentum_buffer")
                    if momentum_buffer is not None:
                        momentum_buffer.mul_(masks[name])

            cifar_loss_sum += loss_cifar.item()
            mini_kl_sum += loss_mini_kl.item()

            cifar_predictions = cifar_logits.argmax(dim=1)
            cifar_correct += (
                cifar_predictions == cifar_labels
            ).sum().item()
            cifar_total += cifar_labels.size(0)
            mini_steps += 1

            progress.set_postfix(
                CE=f"{loss_cifar.item():.4f}",
                KL=f"{loss_mini_kl.item():.4f}",
                CIFAR_train_acc=f"{100.0*cifar_correct/max(cifar_total,1):.2f}%"
            )

        elapsed = time.time() - start_time
        print(
            f"Epoch {epoch + 1}: "
            f"CIFAR_CE={cifar_loss_sum/max(steps_per_epoch,1):.5f} | "
            f"Mini_KL={mini_kl_sum/max(mini_steps,1):.5f} | "
            f"CIFAR train accuracy="
            f"{100.0*cifar_correct/max(cifar_total,1):.3f}% | "
            f"Time={elapsed/60:.2f} min"
        )


# ================================================================
# 10. MAIN DLA ITERATIVE OPTIMIZATION
# ================================================================

results = []

W_current = save_weights(student)
best_cifar_accuracy = evaluate_cifar(student, cifar_val_loader)
best_weights = save_weights(student)
best_iteration = 0
patience_counter = 0

print("\nINITIAL CIFAR-100 VALIDATION ACCURACY:",
      f"{best_cifar_accuracy:.3f}%")
print("INITIAL MINI-IMAGENET VALIDATION ACCURACY:",
      f"{evaluate_mini(student, mini_val_loader):.3f}%")

print("\nSTARTING DLA EXPERIMENT")

for iteration in range(1, NUM_ITERATIONS + 1):
    print("\n" + "=" * 78)
    print(f"ITERATION {iteration}/{NUM_ITERATIONS}")
    print("=" * 78)

    iteration_start = time.time()

    # Restore previous DLA interpolated weights.
    load_weights(student, W_current)

    # W0: beginning weights for this iteration.
    W0 = save_weights(student)

    # Before-training validation measurements.
    cifar_before = evaluate_cifar(student, cifar_val_loader)
    mini_before = evaluate_mini(student, mini_val_loader)

    print("\nBEFORE TRAINING")
    print(f"CIFAR-100 validation accuracy: {cifar_before:.3f}%")
    print(f"MiniImageNet validation accuracy: {mini_before:.3f}%")

    # Generate a new independent random 10% mask per eligible tensor.
    masks, total_eligible, total_selected = generate_masks(student)

    # Joint training: CIFAR CE + MiniImageNet KL.
    train_one_iteration(student, teacher, masks, W0, iteration)

    # W1: trained candidate weights, before DLA interpolation.
    W1 = save_weights(student)

    # Candidate accuracies.
    cifar_candidate = evaluate_cifar(student, cifar_val_loader)
    mini_candidate = evaluate_mini(student, mini_val_loader)

    # ONLY CIFAR-100 accuracy change drives DLA fitness and beta.
    delta_cifar = cifar_candidate - cifar_before
    delta_mini = mini_candidate - mini_before  # reporting only

    fitness = delta_cifar
    beta = 1.0 / (1.0 + np.exp(-np.clip(K * fitness, -60, 60)))

    # Original W0/W1 interpolation, strictly restricted to selected
    # parameter coordinates. Unselected parameters remain exactly W0.
    interpolate_weights_masked(student, W0, W1, masks, beta)

    # Store the interpolated model as the next iteration's starting point.
    W_current = save_weights(student)

    # Verify no unselected parameter changed across this DLA iteration.
    changed_count, all_parameter_count = count_changed_parameters(student, W0)
    if changed_count > total_selected:
        raise RuntimeError(
            "Mask verification failed: changed parameter count exceeds "
            "the selected parameter count."
        )

    # Evaluate the actual interpolated model.
    cifar_after = evaluate_cifar(student, cifar_val_loader)
    mini_after = evaluate_mini(student, mini_val_loader)

    # Patience uses ONLY interpolated CIFAR-100 validation accuracy.
    improved = cifar_after > best_cifar_accuracy + 1e-8

    if improved:
        best_cifar_accuracy = cifar_after
        best_weights = save_weights(student)
        best_iteration = iteration
        patience_counter = 0

        torch.save(
            {
                "iteration": iteration,
                "model_state_dict": student.state_dict(),
                "weights": best_weights,
                "cifar_validation_accuracy": best_cifar_accuracy,
                "mini_validation_accuracy": mini_after,
            },
            os.path.join(OUTPUT_DIR, "best_cifar_checkpoint.pth")
        )
        print("NEW BEST CIFAR-100 CHECKPOINT SAVED.")
    else:
        patience_counter += 1

    elapsed = time.time() - iteration_start

    record = {
        "Iteration": iteration,
        "CIFAR_Before": cifar_before,
        "CIFAR_Candidate_After_Training": cifar_candidate,
        "CIFAR_After_Interpolation": cifar_after,
        "Mini_Before": mini_before,
        "Mini_Candidate_After_Training": mini_candidate,
        "Mini_After_Interpolation": mini_after,
        "Delta_CIFAR_Candidate": delta_cifar,
        "Delta_Mini_Candidate": delta_mini,
        "Fitness_CIFAR_Only": fitness,
        "Beta": beta,
        "W0_Contribution_Percent": (1.0 - beta) * 100.0,
        "W1_Contribution_Percent": beta * 100.0,
        "Selected_Parameters": total_selected,
        "Eligible_Parameters": total_eligible,
        "Changed_Parameters_After_Interpolation": changed_count,
        "CIFAR_Improved_Best": improved,
        "Patience_Counter": patience_counter,
        "Time_Minutes": elapsed / 60.0,
    }
    results.append(record)

    print("\nITERATION RESULTS")
    print(f"CIFAR candidate delta: {delta_cifar:+.4f}%")
    print(f"MiniImageNet candidate delta (report only): {delta_mini:+.4f}%")
    print(f"Fitness (CIFAR only): {fitness:+.5f}")
    print(f"Beta: {beta:.5f}")
    print(f"W0 contribution: {(1.0-beta)*100:.2f}%")
    print(f"W1 contribution: {beta*100:.2f}%")
    print(f"CIFAR after interpolation: {cifar_after:.3f}%")
    print(f"Mini after interpolation: {mini_after:.3f}%")
    print(
        f"Changed parameters: {changed_count:,} "
        f"(selected {total_selected:,})"
    )
    print(f"Best CIFAR accuracy: {best_cifar_accuracy:.3f}%")
    print(f"Patience: {patience_counter}/{PATIENCE}")
    print(f"Iteration time: {elapsed/60:.2f} minutes")

    # Save a checkpoint and CSV after every iteration.
    torch.save(
        {
            "iteration": iteration,
            "model_state_dict": student.state_dict(),
            "W_current": W_current,
            "masks": masks,
            "results": results,
            "best_cifar_accuracy": best_cifar_accuracy,
            "best_iteration": best_iteration,
        },
        os.path.join(OUTPUT_DIR, f"iteration_{iteration:02d}.pth")
    )

    pd.DataFrame(results).to_csv(
        os.path.join(OUTPUT_DIR, "iteration_results.csv"),
        index=False
    )

    # Stop after 3 consecutive iterations without a new best CIFAR score.
    if patience_counter >= PATIENCE:
        print(
            f"\nEARLY STOPPING: CIFAR-100 validation accuracy did not "
            f"improve for {PATIENCE} consecutive iterations."
        )
        break


# ================================================================
# 11. FINAL EVALUATION OF BEST CIFAR CHECKPOINT
# ================================================================

print("\n" + "=" * 78)
print("DLA EXPERIMENT FINISHED")
print("=" * 78)

# Restore best checkpoint according to CIFAR-100 validation accuracy only.
load_weights(student, best_weights)

final_best_cifar = evaluate_cifar(student, cifar_val_loader)
final_best_mini = evaluate_mini(student, mini_val_loader)

print(f"Best iteration: {best_iteration}")
print(f"Best CIFAR-100 validation accuracy: {final_best_cifar:.3f}%")
print(f"MiniImageNet validation accuracy at that checkpoint: "
      f"{final_best_mini:.3f}%")

results_df = pd.DataFrame(results)
results_df.to_csv(
    os.path.join(OUTPUT_DIR, "final_results.csv"),
    index=False
)

torch.save(
    {
        "model_state_dict": student.state_dict(),
        "best_weights": best_weights,
        "best_iteration": best_iteration,
        "best_cifar_validation_accuracy": final_best_cifar,
        "mini_validation_accuracy_at_best_cifar_checkpoint": final_best_mini,
        "results": results,
    },
    os.path.join(OUTPUT_DIR, "final_best_model.pth")
)

print("\nResults saved to:", OUTPUT_DIR)
print("CSV:", os.path.join(OUTPUT_DIR, "final_results.csv"))
print("Best model:", os.path.join(OUTPUT_DIR, "final_best_model.pth"))


DEVICE: cuda
GPU: Tesla T4
Found ImageNet mapping: /kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/imagenet_class_index.json


100%|██████████| 169M/169M [00:07<00:00, 22.4MB/s] 



CIFAR-100 TRAIN: 45000
CIFAR-100 VALIDATION: 5000

MINI-IMAGENET CLASSES: 1000
MINI-IMAGENET TRAIN: 2833
MINI-IMAGENET VALIDATION: 1090
MiniImageNet split: 80% train / 20% validation per class
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 259MB/s]



STUDENT MODEL CREATED
Student parameters: 11740812
Teacher is frozen and used only to generate MiniImageNet soft targets.

INITIAL CIFAR-100 VALIDATION ACCURACY: 0.860%
INITIAL MINI-IMAGENET VALIDATION ACCURACY: 68.899%

STARTING DLA EXPERIMENT

ITERATION 1/40

BEFORE TRAINING
CIFAR-100 validation accuracy: 0.860%
MiniImageNet validation accuracy: 68.899%
  backbone.0.weight                  : selected 940/9,408 (9.99%)
  backbone.1.weight                  : selected 6/64 (9.38%)
  backbone.1.bias                    : selected 6/64 (9.38%)
  backbone.4.0.conv1.weight          : selected 3,686/36,864 (10.00%)
  backbone.4.0.bn1.weight            : selected 6/64 (9.38%)
  backbone.4.0.bn1.bias              : selected 6/64 (9.38%)
  backbone.4.0.conv2.weight          : selected 3,686/36,864 (10.00%)
  backbone.4.0.bn2.weight            : selected 6/64 (9.38%)
  backbone.4.0.bn2.bias              : selected 6/64 (9.38%)
  backbone.4.1.conv1.weight          : selected 3,686/36,864 (10.00%)

Iteration 1 | Epoch 1/5:   0%|          | 0/352 [00:00<?, ?it/s]

... (105 outputs left out)

Iteration 11 | Epoch 4/5:   0%|          | 0/352 [00:00<?, ?it/s]

Epoch 4: CIFAR_CE=0.41499 | Mini_KL=0.04874 | CIFAR train accuracy=87.253% | Time=3.53 min


Iteration 11 | Epoch 5/5:   0%|          | 0/352 [00:00<?, ?it/s]

Epoch 5: CIFAR_CE=0.39750 | Mini_KL=0.04847 | CIFAR train accuracy=87.711% | Time=3.53 min


# chagpt changes

In [ ]:
# DLA ResNet-18: CIFAR-100 + Mini-ImageNet
# Preserves: random 10% mask per parameter tensor, new mask each iteration,
# 5 epochs/iteration, max 40 iterations, patience 3, CE on CIFAR,
# teacher-student KL on MiniImageNet, CIFAR-only beta/early stopping,
# and W0/W1 interpolation only on selected entries.
#
# MiniImageNet metric in this script is agreement with frozen ImageNet
# teacher pseudo-labels, NOT ground-truth MiniImageNet classification accuracy.
# The supplied run has only 2,833 train + 1,090 validation images across
# 1,000 folders; verify this is the intended dataset before interpreting it.

import os, copy, time, csv, math, random
from pathlib import Path
from collections import defaultdict
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CIFAR_ROOT = "/kaggle/input/datasets/melikechan/cifar100"
MINI_ROOT = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/images"
OUT = "/kaggle/working/DLA_ResNet18_results"
os.makedirs(OUT, exist_ok=True)

FRACTION, MAX_ITER, EPOCHS, PATIENCE = 0.10, 40, 5, 3
BATCH_CIFAR, BATCH_MINI, WORKERS = 128, 32, 0
LR, WD, KL_WEIGHT, TEMP = 1e-3, 1e-4, 1.0, 2.0
BETA_SLOPE, BETA_MIN, BETA_MAX = 0.05, 0.50, 0.80
MIN_IMPROVEMENT = 0.02  # percentage points

def seed_all(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
seed_all(SEED)

MEAN, STD = (0.485,0.456,0.406), (0.229,0.224,0.225)
cifar_train_tf = transforms.Compose([
    transforms.Resize(224), transforms.RandomCrop(224, padding=16),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(),
    transforms.Normalize(MEAN, STD)])
cifar_eval_tf = transforms.Compose([
    transforms.Resize(224), transforms.ToTensor(), transforms.Normalize(MEAN, STD)])
mini_train_tf = transforms.Compose([
    transforms.Resize(256), transforms.RandomResizedCrop(224, scale=(0.75,1.0)),
    transforms.RandomHorizontalFlip(), transforms.ToTensor(), transforms.Normalize(MEAN,STD)])
mini_eval_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(224), transforms.ToTensor(),
    transforms.Normalize(MEAN,STD)])

def loader(ds, bs, shuffle):
    return DataLoader(ds, batch_size=bs, shuffle=shuffle, drop_last=shuffle,
        num_workers=WORKERS, pin_memory=DEVICE.type=="cuda",
        persistent_workers=WORKERS>0)

def resolve_cifar_root():
    # torchvision expects <root>/cifar-100-python (or downloads it there).
    candidates = [
        Path(CIFAR_ROOT),
        Path("/kaggle/working/cifar100"),
        Path("/kaggle/working"),
    ]
    for candidate in candidates:
        if (candidate / "cifar-100-python").exists():
            return str(candidate)
        if candidate.name == "cifar-100-python" and candidate.exists():
            return str(candidate.parent)
    # If the Kaggle input dataset is not in torchvision's native format,
    # download the official CIFAR-100 archive into writable working storage.
    return "/kaggle/working/cifar100"

def make_cifar():
    cifar_root = resolve_cifar_root()
    print("Using CIFAR-100 root:", cifar_root)
    tr = datasets.CIFAR100(cifar_root, train=True, download=True, transform=cifar_train_tf)
    ev = datasets.CIFAR100(cifar_root, train=True, download=True, transform=cifar_eval_tf)
    # Read labels directly; do not decode/resize 50,000 images just to split.
    groups = defaultdict(list)
    for i, y in enumerate(tr.targets):
        groups[int(y)].append(i)
    rng = np.random.default_rng(SEED); ti=[]; vi=[]
    for ids in groups.values():
        ids=np.asarray(ids); rng.shuffle(ids); n=max(1, round(.10*len(ids)))
        vi.extend(ids[:n].tolist()); ti.extend(ids[n:].tolist())
    return Subset(tr,ti), Subset(ev,vi)

def make_mini():
    root=Path(MINI_ROOT)
    if not root.exists():
        raise FileNotFoundError(
            f"MiniImageNet folder not found: {root}\n"
            "In Kaggle, attach the MiniImageNet dataset and update MINI_ROOT "
            "to the directory containing class subfolders."
        )
    tr=datasets.ImageFolder(str(root), transform=mini_train_tf)
    ev=datasets.ImageFolder(str(root), transform=mini_eval_tf)
    groups=defaultdict(list)
    for i,(_,y) in enumerate(tr.samples): groups[y].append(i)
    rng=np.random.default_rng(SEED); ti=[]; vi=[]; counts=[]
    for ids in groups.values():
        ids=np.asarray(ids); rng.shuffle(ids); counts.append(len(ids))
        nv=min(max(1,round(.20*len(ids))),len(ids)-1) if len(ids)>=2 else 0
        vi.extend(ids[:nv].tolist()); ti.extend(ids[nv:].tolist())
    print(f"Mini classes={len(groups)}, images={len(tr)}, train={len(ti)}, val={len(vi)}")
    print(f"Images/class min/median/max={min(counts)}/{np.median(counts):.1f}/{max(counts)}")
    if len(groups)!=100: print("WARNING: This is not the usual 100-class MiniImageNet split.")
    return Subset(tr,ti),Subset(ev,vi)

cifar_tr,cifar_va=make_cifar()
mini_tr,mini_va=make_mini()
cl=loader(cifar_tr,BATCH_CIFAR,True); cv=loader(cifar_va,BATCH_CIFAR,False)
ml=loader(mini_tr,BATCH_MINI,True); mv=loader(mini_va,BATCH_MINI,False)

class DualResNet18(nn.Module):
    def __init__(self, pretrained=False):
        super().__init__()
        net=models.resnet18(weights=ResNet18_Weights.DEFAULT if pretrained else None)
        self.backbone=nn.Sequential(*list(net.children())[:-1])
        self.imagenet_head=copy.deepcopy(net.fc)
        self.cifar_head=nn.Linear(512,100)
    def forward(self,x):
        f=self.backbone(x).flatten(1)
        return self.cifar_head(f),self.imagenet_head(f)

student=DualResNet18(pretrained=True).to(DEVICE)
teacher=DualResNet18(pretrained=False).to(DEVICE)
teacher.load_state_dict(student.state_dict())
teacher.eval()
for p in teacher.parameters(): p.requires_grad_(False)

@torch.no_grad()
def eval_cifar(model):
    model.eval(); good=n=0
    for x,y in cv:
        x=x.to(DEVICE); y=y.to(DEVICE); z,_=model(x)
        good+=(z.argmax(1)==y).sum().item(); n+=y.numel()
    return 100*good/max(n,1)

@torch.no_grad()
def eval_mini_agreement(model):
    # Teacher agreement is a proxy/retention metric, not dataset-label accuracy.
    model.eval(); good=n=0
    for x,_ in mv:
        x=x.to(DEVICE); _,s=model(x); _,t=teacher(x)
        good+=(s.argmax(1)==t.argmax(1)).sum().item(); n+=x.size(0)
    return 100*good/max(n,1)

def make_mask(p):
    n=p.numel(); k=max(1,round(FRACTION*n))
    m=torch.zeros(n,dtype=torch.bool,device=p.device)
    m[torch.randperm(n,device=p.device)[:k]]=True
    return m.view_as(p)

def make_masks(model):
    masks={}; total=chosen=0
    for name,p in model.named_parameters():
        masks[name]=make_mask(p); total+=p.numel(); chosen+=masks[name].sum().item()
    print(f"Selected {chosen:,.0f}/{total:,} ({100*chosen/total:.3f}%)")
    return masks

def freeze_bn_stats(model):
    for m in model.modules():
        if isinstance(m,nn.modules.batchnorm._BatchNorm): m.eval()

def train_candidate(start_state,masks):
    cand=DualResNet18(pretrained=False).to(DEVICE)
    cand.load_state_dict(start_state); params=dict(cand.named_parameters())
    opt=torch.optim.SGD(cand.parameters(),lr=LR,momentum=.9,
                        weight_decay=WD,nesterov=True)
    for ep in range(EPOCHS):
        cand.train(); freeze_bn_stats(cand)
        ci=iter(cl); mi=iter(ml); steps=max(len(cl),len(ml))
        ce_sum=kl_sum=0.; correct=seen=0; tic=time.time()
        for _ in range(steps):
            try: x,y=next(ci)
            except StopIteration: ci=iter(cl); x,y=next(ci)
            try: u,_=next(mi)
            except StopIteration: mi=iter(ml); u,_=next(mi)
            x=x.to(DEVICE,non_blocking=True); y=y.to(DEVICE,non_blocking=True)
            u=u.to(DEVICE,non_blocking=True)
            opt.zero_grad(set_to_none=True)
            cz,_=cand(x); _,sz=cand(u)
            with torch.no_grad(): _,tz=teacher(u)
            ce=F.cross_entropy(cz,y)
            kl=F.kl_div(F.log_softmax(sz/TEMP,1),
                        F.softmax(tz/TEMP,1),reduction="batchmean")*(TEMP**2)
            loss=ce+KL_WEIGHT*kl
            loss.backward()
            for name,p in params.items():
                if p.grad is not None: p.grad.mul_(masks[name])
            # Restore unselected values after optimizer step so momentum/decay
            # cannot modify them either.
            with torch.no_grad():
                frozen={name:p[~masks[name]].clone() for name,p in params.items()}
            opt.step()
            with torch.no_grad():
                for name,p in params.items(): p[~masks[name]]=frozen[name]
            ce_sum+=ce.item(); kl_sum+=kl.item()
            correct+=(cz.argmax(1)==y).sum().item(); seen+=y.numel()
        print(f"Epoch {ep+1}/{EPOCHS}: CE={ce_sum/steps:.4f}, "
              f"KL={kl_sum/steps:.4f}, CIFAR train acc={100*correct/max(seen,1):.2f}%, "
              f"time={time.time()-tic:.1f}s")
    return cand

def beta_from_cifar_delta(delta):
    # CIFAR-only control. Reject a candidate if CIFAR validation did not improve.
    if delta<=MIN_IMPROVEMENT: return 0.0
    raw=1/(1+math.exp(-BETA_SLOPE*delta))
    return float(np.clip(raw,BETA_MIN,BETA_MAX))

def interpolate_selected(model,candidate,masks,beta):
    a=dict(model.named_parameters()); b=dict(candidate.named_parameters())
    with torch.no_grad():
        for name,p in a.items():
            m=masks[name]
            p[m]=(1-beta)*p[m]+beta*b[name][m]

print("Device:",DEVICE)
cifar0=eval_cifar(student); mini0=eval_mini_agreement(student)
print(f"Initial CIFAR validation={cifar0:.3f}%")
print(f"Initial Mini teacher agreement={mini0:.3f}%")
best=cifar0; best_state=copy.deepcopy(student.state_dict()); patience=0
csv_path=os.path.join(OUT,"dla_history.csv")
with open(csv_path,"w",newline="") as f:
    csv.writer(f).writerow(["iteration","cifar_before","cifar_candidate","cifar_after",
        "mini_agreement_before","mini_agreement_candidate","mini_agreement_after",
        "cifar_delta_pp","beta","patience"])

for it in range(1,MAX_ITER+1):
    print("\n"+"="*65+f"\nITERATION {it}/{MAX_ITER}\n"+"="*65)
    cb=eval_cifar(student); mb=eval_mini_agreement(student)
    start_state=copy.deepcopy(student.state_dict())
    masks=make_masks(student)
    candidate=train_candidate(start_state,masks)
    cc=eval_cifar(candidate); mc=eval_mini_agreement(candidate)
    delta=cc-cb; beta=beta_from_cifar_delta(delta)
    if beta>0: interpolate_selected(student,candidate,masks,beta)
    else: print("Candidate rejected by CIFAR-only rule; retaining W0.")
    ca=eval_cifar(student); ma=eval_mini_agreement(student)
    if ca>best+MIN_IMPROVEMENT:
        best=ca; best_state=copy.deepcopy(student.state_dict()); patience=0
        torch.save({"state_dict":best_state,"iteration":it,"best_cifar_val":best},
                   os.path.join(OUT,"best_dla_resnet18.pt"))
        print("Saved best CIFAR checkpoint.")
    else: patience+=1
    print(f"CIFAR {cb:.3f} -> candidate {cc:.3f} -> interpolated {ca:.3f}")
    print(f"Mini teacher agreement {mb:.3f} -> candidate {mc:.3f} -> {ma:.3f}")
    print(f"CIFAR delta={delta:+.3f} pp, beta={beta:.4f}, "
          f"best={best:.3f}%, patience={patience}/{PATIENCE}")
    with open(csv_path,"a",newline="") as f:
        csv.writer(f).writerow([it,cb,cc,ca,mb,mc,ma,delta,beta,patience])
    if ca<best: student.load_state_dict(best_state)
    if patience>=PATIENCE:
        print("Early stopping: 3 consecutive iterations without CIFAR improvement.")
        break

student.load_state_dict(best_state)
torch.save(student.state_dict(),os.path.join(OUT,"final_best_model_state.pt"))
print(f"\nFinished. Best CIFAR validation accuracy={best:.3f}%")
print("Mini metric is teacher agreement, not ground-truth MiniImageNet accuracy.")
print("CSV:",csv_path)
print("Output directory:",OUT)


Using CIFAR-100 root: /kaggle/working/cifar100


 25%|██▌       | 42.9M/169M [05:31<16:53, 124kB/s]

## kl divergence for mini

In [1]:
import os
import random
import copy
import glob
import time
import zipfile
import csv
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms

from torchvision.models import resnet18, ResNet18_Weights
from torchvision.datasets import CIFAR100, ImageFolder

from torch.utils.data import DataLoader
from itertools import cycle

# ============================================================
# CONFIGURATION
# ============================================================

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

if DEVICE.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Dataset paths
CIFAR_ROOT = "/kaggle/input/datasets/melikechan/cifar100"

MINI_ROOT = "/kaggle/input/datasets/deeptrial/miniimagenet"

# Training configuration
ITERATIONS = 40
EPOCHS_PER_ITERATION = 5

TOTAL_EPOCHS = ITERATIONS * EPOCHS_PER_ITERATION

PARAMETER_RATIO = 0.10

BATCH_SIZE = 64
NUM_WORKERS = 2

LEARNING_RATE = 0.001
MOMENTUM = 0.9

TEMPERATURE = 2.0
LAMBDA_KL = 1.0

IMAGE_SIZE = 224

OUTPUT_DIR = "/kaggle/working/sparse_resnet18_results"

os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Total epochs:", TOTAL_EPOCHS)
print("Iterations:", ITERATIONS)
print("Epochs per iteration:", EPOCHS_PER_ITERATION)
print("Selected parameter ratio:", PARAMETER_RATIO)

Device: cuda
GPU: Tesla T4
Total epochs: 200
Iterations: 40
Epochs per iteration: 5
Selected parameter ratio: 0.1


In [2]:
def extract_zip_files(root, destination):

    os.makedirs(destination, exist_ok=True)

    for zip_path in glob.glob(
        os.path.join(root, "**", "*.zip"),
        recursive=True
    ):

        print("Extracting:", zip_path)

        with zipfile.ZipFile(zip_path, "r") as z:
            z.extractall(destination)


# Extract if the datasets are supplied as ZIP files
extract_zip_files(
    MINI_ROOT,
    "/kaggle/working/mini_extracted"
)

extract_zip_files(
    CIFAR_ROOT,
    "/kaggle/working/cifar_extracted"
)


def find_imagefolder_directories(root):

    candidates = []

    for current, dirs, files in os.walk(root):

        class_dirs = [
            os.path.join(current, d)
            for d in dirs
        ]

        valid_classes = 0
        total_images = 0

        for class_dir in class_dirs:

            images = glob.glob(
                os.path.join(class_dir, "*")
            )

            images = [
                p for p in images
                if p.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                )
            ]

            if images:
                valid_classes += 1
                total_images += len(images)

        if valid_classes >= 2:

            candidates.append(
                (current, total_images)
            )

    return sorted(
        candidates,
        key=lambda x: x[1],
        reverse=True
    )


def find_split(root, preferred_names):

    candidates = [root]

    for current, dirs, files in os.walk(root):

        for d in dirs:
            candidates.append(os.path.join(current, d))

    for preferred in preferred_names:

        for path in candidates:

            if os.path.basename(path).lower() == preferred:

                results = find_imagefolder_directories(path)

                if results:
                    return results[0][0]

    # Fallback
    results = find_imagefolder_directories(root)

    if results:
        return results[0][0]

    return None


# ============================================================
# CIFAR DATASET
# ============================================================

def load_cifar_dataset():

    roots = [
        CIFAR_ROOT,
        "/kaggle/working/cifar_extracted"
    ]

    for root in roots:

        for current, dirs, files in os.walk(root):

            if "cifar-100-python" in dirs:

                parent = current

                train = CIFAR100(
                    root=parent,
                    train=True,
                    download=False
                )

                test = CIFAR100(
                    root=parent,
                    train=False,
                    download=False
                )

                print("Loaded standard CIFAR-100 dataset")

                return train, test

            if "train" in files and "test" in files:

                if os.path.basename(current) == "cifar-100-python":

                    train = CIFAR100(
                        root=os.path.dirname(current),
                        train=True,
                        download=False
                    )

                    test = CIFAR100(
                        root=os.path.dirname(current),
                        train=False,
                        download=False
                    )

                    return train, test

    train_dir = find_split(
        CIFAR_ROOT,
        ["train"]
    )

    test_dir = find_split(
        CIFAR_ROOT,
        ["test", "val", "validation"]
    )

    if train_dir and test_dir:

        print("Loaded CIFAR-100 using ImageFolder")

        return (
            ImageFolder(train_dir),
            ImageFolder(test_dir)
        )

    raise RuntimeError(
        "CIFAR-100 structure not recognized. "
        "Inspect CIFAR_ROOT and its subdirectories."
    )


# ============================================================
# MINI IMAGENET DATASET
# ============================================================

def load_mini_dataset():

    roots = [
        MINI_ROOT,
        "/kaggle/working/mini_extracted"
    ]

    for root in roots:

        train_dir = find_split(
            root,
            ["tr", "train"]
        )

        val_dir = find_split(
            root,
            ["val", "validation"]
        )

        if train_dir:

            train = ImageFolder(train_dir)

            if val_dir and val_dir != train_dir:

                val = ImageFolder(val_dir)

            else:
                val = None

            print("MiniImageNet train:", train_dir)
            print("MiniImageNet val:", val_dir)

            return train, val

    raise RuntimeError(
        "MiniImageNet structure not recognized. "
        "Inspect MINI_ROOT and its subdirectories."
    )


cifar_train_raw, cifar_test_raw = load_cifar_dataset()

mini_train_raw, mini_val_raw = load_mini_dataset()

print("\nDataset information")

print("CIFAR train:", len(cifar_train_raw))
print("CIFAR test:", len(cifar_test_raw))

print("MiniImageNet train:", len(mini_train_raw))

if mini_val_raw:
    print("MiniImageNet validation:", len(mini_val_raw))

Loaded CIFAR-100 using ImageFolder
MiniImageNet train: /kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/images
MiniImageNet val: /kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/images

Dataset information
CIFAR train: 50000
CIFAR test: 10000
MiniImageNet train: 3923


In [3]:
train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


eval_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])


cifar_train_raw.transform = train_transform
cifar_test_raw.transform = eval_transform

mini_train_raw.transform = train_transform

if mini_val_raw:
    mini_val_raw.transform = eval_transform


cifar_train_loader = DataLoader(
    cifar_train_raw,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

cifar_test_loader = DataLoader(
    cifar_test_raw,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True
)

mini_train_loader = DataLoader(
    mini_train_raw,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    drop_last=True
)

mini_eval_loader = None

if mini_val_raw:

    mini_eval_loader = DataLoader(
        mini_val_raw,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
        pin_memory=True
    )

print("DataLoaders ready.")

DataLoaders ready.


In [4]:
class SparseResNet18(nn.Module):

    def __init__(self):

        super().__init__()

        # Pretrained ResNet-18
        base = resnet18(
            weights=ResNet18_Weights.DEFAULT
        )

        # Shared feature extractor
        self.backbone = nn.Sequential(
            *list(base.children())[:-1]
        )

        # Original ImageNet classifier
        self.imagenet_head = copy.deepcopy(base.fc)

        # New CIFAR-100 classifier
        self.cifar_head = nn.Linear(
            512,
            100
        )

        # Initialize CIFAR head
        nn.init.kaiming_normal_(
            self.cifar_head.weight,
            mode="fan_out",
            nonlinearity="relu"
        )

        nn.init.zeros_(
            self.cifar_head.bias
        )

    def extract_features(self, x):

        features = self.backbone(x)

        features = torch.flatten(
            features,
            1
        )

        return features

    def forward_cifar(self, x):

        features = self.extract_features(x)

        return self.cifar_head(features)

    def forward_imagenet(self, x):

        features = self.extract_features(x)

        return self.imagenet_head(features)


# Original pretrained teacher
teacher = resnet18(
    weights=ResNet18_Weights.DEFAULT
).to(DEVICE)

teacher.eval()

for p in teacher.parameters():
    p.requires_grad = False


# Updated student
student = SparseResNet18().to(DEVICE)

print(student)

print(
    "\nTotal student parameters:",
    sum(p.numel() for p in student.parameters())
)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 184MB/s] 


SparseResNet18(
  (backbone): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
      (1): BasicBlock(
        (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_sta

In [5]:
def generate_random_masks(model, ratio=0.10):

    masks = {}

    total_parameters = 0
    selected_parameters = 0

    for name, param in model.named_parameters():

        total_parameters += param.numel()

        # Fresh random selection for this iteration
        mask = (
            torch.rand_like(param) < ratio
        )

        masks[name] = mask

        selected_parameters += mask.sum().item()

    selected_ratio = (
        selected_parameters / total_parameters
    ) * 100

    print(
        f"Selected parameters: "
        f"{selected_parameters:,} / "
        f"{total_parameters:,}"
    )

    print(
        f"Actual selection ratio: "
        f"{selected_ratio:.2f}%"
    )

    return masks


def apply_gradient_masks(model, masks):

    for name, param in model.named_parameters():

        if param.grad is not None:

            param.grad.mul_(
                masks[name]
            )


def freeze_batchnorm_statistics(model):

    for module in model.modules():

        if isinstance(
            module,
            nn.BatchNorm2d
        ):

            module.eval()

In [6]:
criterion_ce = nn.CrossEntropyLoss()


def calculate_kl_loss(
    student_logits,
    teacher_logits,
    temperature=2.0
):

    student_log_probs = F.log_softmax(
        student_logits / temperature,
        dim=1
    )

    teacher_probs = F.softmax(
        teacher_logits / temperature,
        dim=1
    )

    kl_loss = F.kl_div(
        student_log_probs,
        teacher_probs,
        reduction="batchmean"
    )

    return kl_loss * (temperature ** 2)

In [7]:
@torch.no_grad()
def evaluate_cifar(model):

    model.eval()

    correct = 0
    total = 0

    total_loss = 0

    for images, labels in cifar_test_loader:

        images = images.to(DEVICE)
        labels = labels.to(DEVICE)

        logits = model.forward_cifar(images)

        loss = criterion_ce(
            logits,
            labels
        )

        predictions = logits.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)

        total_loss += loss.item() * labels.size(0)

    accuracy = 100 * correct / total

    avg_loss = total_loss / total

    return accuracy, avg_loss


@torch.no_grad()
def evaluate_mini(model):

    if mini_eval_loader is None:
        return None, None

    model.eval()
    teacher.eval()

    total_kl = 0
    agreement = 0
    total = 0

    for images, _ in mini_eval_loader:

        images = images.to(DEVICE)

        teacher_logits = teacher(images)

        student_logits = model.forward_imagenet(
            images
        )

        kl = calculate_kl_loss(
            student_logits,
            teacher_logits,
            TEMPERATURE
        )

        teacher_pred = teacher_logits.argmax(dim=1)

        student_pred = student_logits.argmax(dim=1)

        agreement += (
            teacher_pred == student_pred
        ).sum().item()

        total_kl += kl.item() * images.size(0)

        total += images.size(0)

    avg_kl = total_kl / total

    agreement_percentage = (
        100 * agreement / total
    )

    return avg_kl, agreement_percentage

In [8]:
from torch.amp import autocast, GradScaler

scaler = GradScaler(
    "cuda",
    enabled=(DEVICE.type == "cuda")
)

history = []

global_epoch = 0

start_time = time.time()


for iteration in range(1, ITERATIONS + 1):

    print("\n" + "=" * 65)

    print(
        f"ITERATION {iteration}/{ITERATIONS}"
    )

    print("=" * 65)

    # --------------------------------------------------------
    # 1. NEW RANDOM 10% PARAMETER SELECTION
    # --------------------------------------------------------

    masks = generate_random_masks(
        student,
        PARAMETER_RATIO
    )

    # --------------------------------------------------------
    # 2. NEW OPTIMIZER FOR THIS ITERATION
    # --------------------------------------------------------

    optimizer = torch.optim.SGD(

        student.parameters(),

        lr=LEARNING_RATE,

        momentum=MOMENTUM,

        weight_decay=0.0
    )

    # --------------------------------------------------------
    # 3. FIVE EPOCHS
    # --------------------------------------------------------

    for epoch in range(1, EPOCHS_PER_ITERATION + 1):

        global_epoch += 1

        student.train()

        # Keep BN running statistics unchanged
        freeze_batchnorm_statistics(student)

        mini_iterator = iter(mini_train_loader)

        total_loss = 0
        total_ce = 0
        total_kl = 0

        correct = 0
        total = 0

        epoch_start = time.time()

        for cifar_images, cifar_labels in cifar_train_loader:

            try:

                mini_images, _ = next(mini_iterator)

            except StopIteration:

                mini_iterator = iter(mini_train_loader)

                mini_images, _ = next(mini_iterator)

            cifar_images = cifar_images.to(
                DEVICE,
                non_blocking=True
            )

            cifar_labels = cifar_labels.to(
                DEVICE,
                non_blocking=True
            )

            mini_images = mini_images.to(
                DEVICE,
                non_blocking=True
            )

            optimizer.zero_grad(set_to_none=True)

            # ------------------------------------------------
            # TEACHER PREDICTIONS ON MINI IMAGENET
            # ------------------------------------------------

            with torch.no_grad():

                with autocast(
                    device_type="cuda",
                    enabled=(DEVICE.type == "cuda")
                ):

                    teacher_logits = teacher(
                        mini_images
                    )

            # ------------------------------------------------
            # STUDENT FORWARD PASS
            # ------------------------------------------------

            with autocast(
                device_type="cuda",
                enabled=(DEVICE.type == "cuda")
            ):

                # CIFAR-100 prediction
                cifar_logits = student.forward_cifar(
                    cifar_images
                )

                # MiniImageNet prediction
                mini_logits = student.forward_imagenet(
                    mini_images
                )

                # Cross Entropy
                loss_ce = criterion_ce(
                    cifar_logits,
                    cifar_labels
                )

                # KL Divergence
                loss_kl = calculate_kl_loss(
                    mini_logits.float(),
                    teacher_logits.float(),
                    TEMPERATURE
                )

                # Combined objective
                loss = (
                    loss_ce +
                    LAMBDA_KL * loss_kl
                )

            # ------------------------------------------------
            # BACKPROPAGATION
            # ------------------------------------------------

            scaler.scale(loss).backward()

            scaler.unscale_(optimizer)

            # Only selected 10% parameters receive updates
            apply_gradient_masks(
                student,
                masks
            )

            scaler.step(optimizer)

            scaler.update()

            # ------------------------------------------------
            # METRICS
            # ------------------------------------------------

            predictions = cifar_logits.argmax(dim=1)

            correct += (
                predictions == cifar_labels
            ).sum().item()

            total += cifar_labels.size(0)

            total_loss += loss.item()

            total_ce += loss_ce.item()

            total_kl += loss_kl.item()

        train_accuracy = 100 * correct / total

        print(
            f"Iteration {iteration:02d} | "
            f"Epoch {epoch:02d}/{EPOCHS_PER_ITERATION} | "
            f"Global Epoch {global_epoch:03d} | "
            f"Loss: {total_loss / len(cifar_train_loader):.4f} | "
            f"CE: {total_ce / len(cifar_train_loader):.4f} | "
            f"KL: {total_kl / len(cifar_train_loader):.4f} | "
            f"Train Acc: {train_accuracy:.2f}% | "
            f"Time: {time.time() - epoch_start:.1f}s"
        )

    # --------------------------------------------------------
    # 4. EVALUATION AFTER EVERY ITERATION
    # --------------------------------------------------------

    cifar_accuracy, cifar_loss = evaluate_cifar(
        student
    )

    mini_kl, mini_agreement = evaluate_mini(
        student
    )

    if mini_kl is None:

        mini_kl = float("nan")
        mini_agreement = float("nan")

    print("\nITERATION EVALUATION")

    print(
        f"CIFAR-100 Accuracy: {cifar_accuracy:.2f}%"
    )

    print(
        f"CIFAR-100 Loss: {cifar_loss:.4f}"
    )

    print(
        f"MiniImageNet KL: {mini_kl:.6f}"
    )

    print(
        f"MiniImageNet Teacher Agreement: "
        f"{mini_agreement:.2f}%"
    )

    # --------------------------------------------------------
    # 5. SAVE HISTORY
    # --------------------------------------------------------

    history.append({

        "Iteration": iteration,

        "Global_Epoch": global_epoch,

        "CIFAR_Accuracy": cifar_accuracy,

        "CIFAR_Loss": cifar_loss,

        "Mini_KL": mini_kl,

        "Mini_Agreement": mini_agreement

    })

    pd.DataFrame(history).to_csv(
        os.path.join(
            OUTPUT_DIR,
            "training_history.csv"
        ),
        index=False
    )

    # --------------------------------------------------------
    # 6. SAVE CHECKPOINT
    # --------------------------------------------------------

    checkpoint_path = os.path.join(
        OUTPUT_DIR,
        f"checkpoint_iteration_{iteration}.pth"
    )

    torch.save({

        "iteration": iteration,

        "global_epoch": global_epoch,

        "student_state_dict": student.state_dict(),

        "teacher_state_dict": teacher.state_dict(),

        "history": history,

        "config": {

            "parameter_ratio": PARAMETER_RATIO,

            "lambda_kl": LAMBDA_KL,

            "temperature": TEMPERATURE,

            "epochs_per_iteration": EPOCHS_PER_ITERATION

        }

    }, checkpoint_path)

    print(
        "\nCheckpoint saved:",
        checkpoint_path
    )


print("\nTRAINING COMPLETED")

print(
    "Total training time:",
    (time.time() - start_time) / 60,
    "minutes"
)


ITERATION 1/40
Selected parameters: 1,172,078 / 11,740,812
Actual selection ratio: 9.98%


KeyboardInterrupt: 

In [ ]:
history_df = pd.DataFrame(history)

print(history_df.tail())

# CIFAR Accuracy
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["Iteration"],
    history_df["CIFAR_Accuracy"],
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("CIFAR-100 Accuracy (%)")

plt.title("CIFAR-100 Accuracy Across Iterations")

plt.grid()

plt.savefig(
    os.path.join(OUTPUT_DIR, "cifar_accuracy.png")
)

plt.show()


# CIFAR Loss
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["Iteration"],
    history_df["CIFAR_Loss"],
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("Cross Entropy Loss")

plt.title("CIFAR-100 Loss Across Iterations")

plt.grid()

plt.savefig(
    os.path.join(OUTPUT_DIR, "cifar_loss.png")
)

plt.show()


# MiniImageNet KL
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["Iteration"],
    history_df["Mini_KL"],
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("KL Divergence")

plt.title("MiniImageNet Knowledge Preservation")

plt.grid()

plt.savefig(
    os.path.join(OUTPUT_DIR, "mini_kl.png")
)

plt.show()


# MiniImageNet Teacher Agreement
plt.figure(figsize=(10, 5))

plt.plot(
    history_df["Iteration"],
    history_df["Mini_Agreement"],
    marker="o"
)

plt.xlabel("Iteration")
plt.ylabel("Teacher Agreement (%)")

plt.title("MiniImageNet Prediction Agreement")

plt.grid()

plt.savefig(
    os.path.join(OUTPUT_DIR, "mini_agreement.png")
)

plt.show()


print("\nFinal CIFAR Accuracy:",
      history_df.iloc[-1]["CIFAR_Accuracy"])

print("\nFinal MiniImageNet KL:",
      history_df.iloc[-1]["Mini_KL"])

print("\nAll results saved in:", OUTPUT_DIR)

## compltet code of above

In [17]:
# ================================================================
# RESNET-18 RANDOM 10% PARAMETER SELECTION
#
# CIFAR-100: Training
# MiniImageNet: Evaluation only
#
# 10% independently selected from EACH:
# Layer 1, Layer 2, Layer 3, Layer 4
#
# 5 epochs per iteration
# Maximum 40 iterations
# Early stopping patience = 3
# ================================================================


import os
import copy
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from tqdm.auto import tqdm


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CIFAR_PATH = (
    "/kaggle/input/datasets/melikechan/cifar100/cifar100"
)

MINI_PATH = (
    "/kaggle/input/datasets/deeptrial/"
    "miniimagenet/ImageNet-Mini/images"
)

OUTPUT_DIR = "/kaggle/working/resnet18_experiment"

os.makedirs(OUTPUT_DIR, exist_ok=True)


BATCH_SIZE = 64

IMAGE_SIZE = 224

EPOCHS_PER_ITERATION = 5

MAX_ITERATIONS = 40

SELECTION_PERCENTAGE = 0.10

PATIENCE = 3

BACKBONE_LR = 0.0001

CLASSIFIER_LR = 0.01

NUM_WORKERS = 2

USE_AMP = DEVICE.type == "cuda"


# Reproducibility

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 70)
print("RESNET-18 RANDOM PARAMETER SELECTION EXPERIMENT")
print("=" * 70)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Selection percentage:", SELECTION_PERCENTAGE * 100, "%")
print("Epochs per iteration:", EPOCHS_PER_ITERATION)
print("Maximum iterations:", MAX_ITERATIONS)


# ================================================================
# 2. DATA TRANSFORMS
# ================================================================

train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


test_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


# ================================================================
# 3. LOAD CIFAR-100
# ================================================================

train_dir = os.path.join(CIFAR_PATH, "train")

test_dir = os.path.join(CIFAR_PATH, "test")


if not os.path.isdir(train_dir):
    raise FileNotFoundError(train_dir)

if not os.path.isdir(test_dir):
    raise FileNotFoundError(test_dir)


train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=test_transform
)


if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("CIFAR train/test class mappings differ.")


if len(train_dataset.classes) != 100:
    raise ValueError("Expected 100 CIFAR classes.")


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


print("\nCIFAR-100 loaded")

print("Train images:", len(train_dataset))

print("Test images:", len(test_dataset))

print("Classes:", len(train_dataset.classes))


# ================================================================
# 4. LOAD MINIIMAGENET
# ================================================================

if not os.path.isdir(MINI_PATH):
    raise FileNotFoundError(MINI_PATH)


mini_dataset = datasets.ImageFolder(
    MINI_PATH,
    transform=test_transform
)


mini_loader = DataLoader(
    mini_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=True
)


print("\nMiniImageNet loaded")

print("Images:", len(mini_dataset))

print("Classes:", len(mini_dataset.classes))


# ================================================================
# 5. LOAD ORIGINAL PRETRAINED RESNET-18
# ================================================================

print("\nLoading pretrained ImageNet ResNet-18...")


original_model = models.resnet18(
    weights=ResNet18_Weights.DEFAULT
).to(DEVICE)


original_model.eval()


for param in original_model.parameters():
    param.requires_grad = False


# ================================================================
# 6. CREATE STUDENT MODEL
# ================================================================

class ResNet18CIFAR(nn.Module):

    def __init__(self, pretrained_model):

        super().__init__()

        # Copy pretrained ResNet-18

        self.backbone = copy.deepcopy(pretrained_model)

        # Preserve original ImageNet classifier

        self.imagenet_probe = copy.deepcopy(
            pretrained_model.fc
        )

        # Remove FC from backbone

        self.backbone.fc = nn.Identity()

        # New CIFAR-100 classifier

        self.cifar_classifier = nn.Linear(
            512,
            100
        )

        # Freeze everything initially

        for param in self.backbone.parameters():
            param.requires_grad = False

        for param in self.imagenet_probe.parameters():
            param.requires_grad = False

        # IMPORTANT:
        # Only Layer 1, Layer 2, Layer 3, Layer 4
        # are eligible for random parameter selection.

        for stage in [
            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4
        ]:

            for param in stage.parameters():
                param.requires_grad = True

        # CIFAR classifier is fully trainable

        for param in self.cifar_classifier.parameters():
            param.requires_grad = True


    def forward(self, x):

        features = self.backbone(x)

        return self.cifar_classifier(features)


    def imagenet_forward(self, x):

        features = self.backbone(x)

        return self.imagenet_probe(features)


student_model = ResNet18CIFAR(
    original_model
).to(DEVICE)


print("\nStudent model created.")


# ================================================================
# 7. DISPLAY PARAMETER COUNTS PER STAGE
# ================================================================

stages = {

    "Layer 1": student_model.backbone.layer1,

    "Layer 2": student_model.backbone.layer2,

    "Layer 3": student_model.backbone.layer3,

    "Layer 4": student_model.backbone.layer4

}


print("\n" + "=" * 70)

print("PARAMETER DISTRIBUTION")

print("=" * 70)


stage_parameter_counts = {}


for name, stage in stages.items():

    count = sum(
        p.numel()
        for p in stage.parameters()
    )

    stage_parameter_counts[name] = count

    print(
        f"{name}: {count:,} parameters | "
        f"10% = {max(1, int(count * 0.10)):,}"
    )


total_stage_parameters = sum(
    stage_parameter_counts.values()
)


classifier_parameters = sum(
    p.numel()
    for p in student_model.cifar_classifier.parameters()
)


print("-" * 70)

print(
    "Total Layer 1-4 parameters:",
    f"{total_stage_parameters:,}"
)

print(
    "CIFAR classifier parameters:",
    f"{classifier_parameters:,}"
)

print(
    "Total eligible parameters:",
    f"{total_stage_parameters + classifier_parameters:,}"
)


# ================================================================
# 8. RANDOM 10% SELECTION IN EACH RESNET STAGE
# ================================================================

current_masks = {}


def generate_stage_masks(model):

    global current_masks

    current_masks = {}

    selected_counts = {}

    for stage_name in [
        "layer1",
        "layer2",
        "layer3",
        "layer4"
    ]:

        stage = getattr(model.backbone, stage_name)

        parameters = list(
            stage.named_parameters()
        )

        total_parameters = sum(
            param.numel()
            for _, param in parameters
        )

        number_selected = max(
            1,
            int(total_parameters * SELECTION_PERCENTAGE)
        )

        # Generate independent random selection
        # across ALL parameters belonging to this stage.

        random_indices = torch.randperm(
            total_parameters,
            device=DEVICE
        )[:number_selected]

        random_indices = torch.sort(
            random_indices
        )[0]

        # Build masks for individual parameter tensors.

        offset = 0

        for parameter_name, param in parameters:

            number = param.numel()

            local_indices = random_indices[
                (random_indices >= offset) &
                (random_indices < offset + number)
            ] - offset

            mask = torch.zeros(
                number,
                device=DEVICE,
                dtype=torch.bool
            )

            if local_indices.numel() > 0:

                mask[local_indices] = True

            mask = mask.reshape(param.shape)

            full_name = (
                f"backbone.{stage_name}.{parameter_name}"
            )

            current_masks[full_name] = mask

            offset += number


        selected_counts[stage_name] = (
            number_selected,
            total_parameters
        )


    print("\nNEW RANDOM PARAMETER SELECTION")

    for name, (selected, total) in selected_counts.items():

        print(
            f"{name}: {selected:,} / {total:,} "
            f"({100 * selected / total:.2f}%)"
        )

    return current_masks


# ================================================================
# 9. GRADIENT MASK HOOKS
# ================================================================

def register_gradient_hooks(model):

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        # Only apply masks to ResNet Layer 1-4.

        if not name.startswith("backbone.layer"):
            continue


        def mask_gradient(gradient, parameter_name=name):

            mask = current_masks[parameter_name]

            return gradient * mask


        param.register_hook(mask_gradient)


register_gradient_hooks(student_model)


# ================================================================
# 10. FREEZE BATCH NORMALIZATION RUNNING STATISTICS
# ================================================================

def freeze_batchnorm_statistics(model):

    for module in model.modules():

        if isinstance(module, nn.BatchNorm2d):

            module.eval()


# ================================================================
# 11. OPTIMIZER
# ================================================================

backbone_parameters = []

for stage_name in [
    "layer1",
    "layer2",
    "layer3",
    "layer4"
]:

    stage = getattr(
        student_model.backbone,
        stage_name
    )

    backbone_parameters.extend(
        stage.parameters()
    )


optimizer = optim.SGD(

    [
        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR
        },

        {
            "params": student_model.cifar_classifier.parameters(),
            "lr": CLASSIFIER_LR
        }
    ],

    momentum=0.0,

    weight_decay=0.0
)


# No momentum and no weight decay:
# Unselected parameters cannot change through optimizer state.


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ================================================================
# 12. CIFAR-100 EVALUATION
# ================================================================

def evaluate_cifar(model):

    model.eval()

    correct = 0

    total = 0

    total_loss = 0.0


    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                outputs = model(images)

                loss = F.cross_entropy(
                    outputs,
                    labels
                )


            total_loss += (
                loss.item() * labels.size(0)
            )

            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)


    accuracy = 100.0 * correct / total

    average_loss = total_loss / total

    return accuracy, average_loss


# ================================================================
# 13. ORIGINAL MINIIMAGENET PREDICTIONS
# ================================================================

print("\nComputing original MiniImageNet predictions...")


original_mini_logits = []


with torch.no_grad():

    for images, _ in tqdm(mini_loader):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = original_model(images)


        original_mini_logits.append(
            outputs.float().cpu()
        )


original_mini_logits = torch.cat(
    original_mini_logits,
    dim=0
)


print(
    "Original predictions:",
    original_mini_logits.shape
)


# ================================================================
# 14. MINIIMAGENET KL DIVERGENCE
# ================================================================

def evaluate_miniimagenet(model):

    model.eval()

    total_kl = 0.0

    total_agreement = 0

    total_images = 0

    start_index = 0


    with torch.no_grad():

        for images, _ in mini_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            batch_size = images.size(0)


            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                updated_logits = model.imagenet_forward(
                    images
                )


            updated_logits = updated_logits.float()

            original_logits = original_mini_logits[
                start_index:start_index + batch_size
            ].to(DEVICE)


            original_probabilities = F.softmax(
                original_logits,
                dim=1
            )

            updated_log_probabilities = F.log_softmax(
                updated_logits,
                dim=1
            )


            # KL(P_original || P_updated)

            kl = F.kl_div(

                updated_log_probabilities,

                original_probabilities,

                reduction="sum"

            )


            total_kl += kl.item()


            original_predictions = original_logits.argmax(
                dim=1
            )

            updated_predictions = updated_logits.argmax(
                dim=1
            )


            total_agreement += (
                original_predictions == updated_predictions
            ).sum().item()


            total_images += batch_size

            start_index += batch_size


    average_kl = total_kl / total_images

    agreement = (
        100.0 * total_agreement / total_images
    )


    return average_kl, agreement


# ================================================================
# 15. TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(model):

    model.train()

    freeze_batchnorm_statistics(model)

    total_loss = 0.0

    correct = 0

    total = 0


    for images, labels in tqdm(
        train_loader,
        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(set_to_none=True)


        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = model(images)

            loss = F.cross_entropy(
                outputs,
                labels
            )


        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()


        total_loss += (
            loss.item() * labels.size(0)
        )

        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


    return (
        total_loss / total,
        100.0 * correct / total
    )


# ================================================================
# 16. INITIAL EVALUATION
# ================================================================

print("\n" + "=" * 70)

print("INITIAL EVALUATION")

print("=" * 70)


initial_cifar_accuracy, initial_cifar_loss = evaluate_cifar(
    student_model
)


initial_mini_kl, initial_mini_agreement = evaluate_miniimagenet(
    student_model
)


print(
    f"Initial CIFAR Accuracy: {initial_cifar_accuracy:.2f}%"
)

print(
    f"Initial CIFAR Loss: {initial_cifar_loss:.4f}"
)

print(
    f"Initial MiniImageNet KL: {initial_mini_kl:.8f}"
)

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)


# ================================================================
# 17. ITERATIVE TRAINING
# ================================================================

best_accuracy = initial_cifar_accuracy

best_model_state = copy.deepcopy(
    student_model.state_dict()
)

patience_counter = 0

history = []


print("\n" + "=" * 70)

print("STARTING TRAINING")

print("=" * 70)


for iteration in range(
    1,
    MAX_ITERATIONS + 1
):

    print("\n")

    print("=" * 70)

    print(
        f"ITERATION {iteration}/{MAX_ITERATIONS}"
    )

    print("=" * 70)


    # ------------------------------------------------------------
    # STEP 1: FRESH 10% SELECTION IN EACH STAGE
    # ------------------------------------------------------------

    generate_stage_masks(student_model)


    # ------------------------------------------------------------
    # STEP 2: CIFAR-100 TRAINING FOR 5 EPOCHS
    # ------------------------------------------------------------

    for epoch in range(
        1,
        EPOCHS_PER_ITERATION + 1
    ):

        train_loss, train_accuracy = train_one_epoch(
            student_model
        )


        print(
            f"Iteration {iteration} | "
            f"Epoch {epoch}/{EPOCHS_PER_ITERATION} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}%"
        )


    # ------------------------------------------------------------
    # STEP 3: CIFAR-100 EVALUATION
    # ------------------------------------------------------------

    cifar_accuracy, cifar_loss = evaluate_cifar(
        student_model
    )


    # ------------------------------------------------------------
    # STEP 4: MINIIMAGENET EVALUATION
    # ------------------------------------------------------------

    mini_kl, mini_agreement = evaluate_miniimagenet(
        student_model
    )


    # ------------------------------------------------------------
    # STEP 5: DISPLAY RESULTS
    # ------------------------------------------------------------

    print("\nITERATION RESULTS")

    print(
        f"CIFAR Accuracy: {cifar_accuracy:.2f}%"
    )

    print(
        f"CIFAR Loss: {cifar_loss:.4f}"
    )

    print(
        f"MiniImageNet KL: {mini_kl:.8f}"
    )

    print(
        f"MiniImageNet Agreement: {mini_agreement:.2f}%"
    )


    # ------------------------------------------------------------
    # STEP 6: STORE HISTORY
    # ------------------------------------------------------------

    history.append({

        "Iteration": iteration,

        "CIFAR_Accuracy": cifar_accuracy,

        "CIFAR_Loss": cifar_loss,

        "MiniImageNet_KL": mini_kl,

        "MiniImageNet_Agreement": mini_agreement

    })


    # ------------------------------------------------------------
    # STEP 7: SAVE BEST MODEL
    # ------------------------------------------------------------

    if cifar_accuracy > best_accuracy:

        best_accuracy = cifar_accuracy

        best_model_state = copy.deepcopy(
            student_model.state_dict()
        )

        patience_counter = 0


        torch.save(

            best_model_state,

            os.path.join(
                OUTPUT_DIR,
                "best_model.pth"
            )

        )


        print("\nNew best CIFAR-100 model saved.")


    else:

        patience_counter += 1

        print(
            f"No improvement: "
            f"{patience_counter}/{PATIENCE}"
        )


    # ------------------------------------------------------------
    # STEP 8: SAVE RUNNING HISTORY
    # ------------------------------------------------------------

    pd.DataFrame(history).to_csv(

        os.path.join(
            OUTPUT_DIR,
            "training_history.csv"
        ),

        index=False

    )


    # ------------------------------------------------------------
    # STEP 9: EARLY STOPPING
    # ------------------------------------------------------------

    if patience_counter >= PATIENCE:

        print("\nEARLY STOPPING TRIGGERED")

        print(
            "No CIFAR accuracy improvement for "
            f"{PATIENCE} consecutive iterations."
        )

        break


# ================================================================
# 18. LOAD BEST MODEL
# ================================================================

student_model.load_state_dict(
    best_model_state
)


# ================================================================
# 19. FINAL EVALUATION
# ================================================================

print("\n" + "=" * 70)

print("FINAL EVALUATION")

print("=" * 70)


final_cifar_accuracy, final_cifar_loss = evaluate_cifar(
    student_model
)


final_mini_kl, final_mini_agreement = evaluate_miniimagenet(
    student_model
)


print("\nRESULTS")

print("-" * 50)

print(
    f"Initial CIFAR Accuracy: "
    f"{initial_cifar_accuracy:.2f}%"
)

print(
    f"Final CIFAR Accuracy: "
    f"{final_cifar_accuracy:.2f}%"
)

print(
    f"Best CIFAR Accuracy: "
    f"{best_accuracy:.2f}%"
)

print(
    f"Initial MiniImageNet KL: "
    f"{initial_mini_kl:.8f}"
)

print(
    f"Final MiniImageNet KL: "
    f"{final_mini_kl:.8f}"
)

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)

print(
    f"Final MiniImageNet Agreement: "
    f"{final_mini_agreement:.2f}%"
)


# ================================================================
# 20. SAVE FINAL MODEL
# ================================================================

torch.save(

    student_model.state_dict(),

    os.path.join(
        OUTPUT_DIR,
        "final_model.pth"
    )

)


pd.DataFrame(history).to_csv(

    os.path.join(
        OUTPUT_DIR,
        "final_training_history.csv"
    ),

    index=False

)


print("\nSaved outputs:")

print(OUTPUT_DIR)

print("\nEXPERIMENT COMPLETED SUCCESSFULLY")

RESNET-18 RANDOM PARAMETER SELECTION EXPERIMENT
Device: cuda
Image size: 224
Batch size: 64
Selection percentage: 10.0 %
Epochs per iteration: 5
Maximum iterations: 40

CIFAR-100 loaded
Train images: 50000
Test images: 10000
Classes: 100

MiniImageNet loaded
Images: 3923
Classes: 1000

Loading pretrained ImageNet ResNet-18...

Student model created.

PARAMETER DISTRIBUTION
Layer 1: 147,968 parameters | 10% = 14,796
Layer 2: 525,568 parameters | 10% = 52,556
Layer 3: 2,099,712 parameters | 10% = 209,971
Layer 4: 8,393,728 parameters | 10% = 839,372
----------------------------------------------------------------------
Total Layer 1-4 parameters: 11,166,976
CIFAR classifier parameters: 51,300
Total eligible parameters: 11,218,276

Computing original MiniImageNet predictions...


  0%|          | 0/62 [00:00<?, ?it/s]

Original predictions: torch.Size([3923, 1000])

INITIAL EVALUATION
Initial CIFAR Accuracy: 0.79%
Initial CIFAR Loss: 4.8382
Initial MiniImageNet KL: 0.00000000
Initial MiniImageNet Agreement: 100.00%

STARTING TRAINING


ITERATION 1/40

NEW RANDOM PARAMETER SELECTION
layer1: 14,796 / 147,968 (10.00%)
layer2: 52,556 / 525,568 (10.00%)
layer3: 209,971 / 2,099,712 (10.00%)
layer4: 839,372 / 8,393,728 (10.00%)


  0%|          | 0/782 [00:00<?, ?it/s]

KeyboardInterrupt: 

# with acc print

In [ ]:
# ================================================================
# RESNET-18 RANDOM PARAMETER SELECTION EXPERIMENT
#
# CIFAR-100: Training
# MiniImageNet: Evaluation only
#
# Independently select 10% parameters from:
# Layer 1, Layer 2, Layer 3, Layer 4
#
# MiniImageNet evaluation:
# 1. Classification Accuracy
# 2. KL Divergence
# 3. Prediction Agreement
# ================================================================


import os
import copy
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader

from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from tqdm.auto import tqdm


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

CIFAR_PATH = (
    "/kaggle/input/datasets/melikechan/cifar100/cifar100"
)

MINI_PATH = (
    "/kaggle/input/datasets/deeptrial/"
    "miniimagenet/ImageNet-Mini/images"
)

IMAGENET_JSON = (
    "/kaggle/input/datasets/deeptrial/"
    "miniimagenet/ImageNet-Mini/imagenet_class_index.json"
)

OUTPUT_DIR = "/kaggle/working/resnet18_experiment"

os.makedirs(OUTPUT_DIR, exist_ok=True)


BATCH_SIZE = 64

IMAGE_SIZE = 224

EPOCHS_PER_ITERATION = 5

MAX_ITERATIONS = 40

SELECTION_PERCENTAGE = 0.10

PATIENCE = 3

BACKBONE_LR = 0.0001

CLASSIFIER_LR = 0.01

NUM_WORKERS = 2

USE_AMP = DEVICE.type == "cuda"


# Reproducibility

random.seed(SEED)

np.random.seed(SEED)

torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 70)
print("RESNET-18 RANDOM PARAMETER SELECTION EXPERIMENT")
print("=" * 70)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)
print("Selection:", SELECTION_PERCENTAGE * 100, "% per stage")
print("Epochs per iteration:", EPOCHS_PER_ITERATION)
print("Maximum iterations:", MAX_ITERATIONS)


# ================================================================
# 2. TRANSFORMS
# ================================================================

train_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.RandomHorizontalFlip(),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


test_transform = transforms.Compose([

    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),

    transforms.ToTensor(),

    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )

])


# ================================================================
# 3. LOAD CIFAR-100
# ================================================================

train_dir = os.path.join(CIFAR_PATH, "train")

test_dir = os.path.join(CIFAR_PATH, "test")


if not os.path.isdir(train_dir):
    raise FileNotFoundError(train_dir)

if not os.path.isdir(test_dir):
    raise FileNotFoundError(test_dir)


train_dataset = datasets.ImageFolder(
    train_dir,
    transform=train_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=test_transform
)


if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("CIFAR train/test class mappings differ.")


if len(train_dataset.classes) != 100:
    raise ValueError("Expected 100 CIFAR classes.")


train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)


test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)


print("\nCIFAR-100 loaded successfully")

print("Training images:", len(train_dataset))
print("Testing images:", len(test_dataset))
print("Classes:", len(train_dataset.classes))


# ================================================================
# 4. LOAD IMAGENET CLASS INDEX
# ================================================================

with open(IMAGENET_JSON, "r") as f:
    imagenet_class_index = json.load(f)


# Convert:
# n01440764 -> 0
# n01443537 -> 1
# etc.

wnid_to_idx = {

    value[0]: int(key)

    for key, value in imagenet_class_index.items()

}


print("\nImageNet class mapping loaded")

print("ImageNet classes:", len(wnid_to_idx))


# ================================================================
# 5. MINIIMAGENET LABEL MAPPER
# ================================================================

class ImageNetLabelMapper:

    def __init__(self, folder_classes, mapping):

        missing = [
            name for name in folder_classes
            if name not in mapping
        ]

        if missing:

            raise ValueError(
                f"Missing ImageNet mappings: {missing[:10]}"
            )

        self.class_indices = [

            mapping[class_name]

            for class_name in folder_classes

        ]


    def __call__(self, folder_target):

        return self.class_indices[folder_target]


# ================================================================
# 6. LOAD MINIIMAGENET
# ================================================================

if not os.path.isdir(MINI_PATH):
    raise FileNotFoundError(MINI_PATH)


# First create dataset to get folder class ordering.

mini_dataset = datasets.ImageFolder(
    MINI_PATH,
    transform=test_transform
)


# Verify class mapping.

if len(mini_dataset.classes) != 1000:
    raise ValueError(
        f"Expected 1000 classes, found {len(mini_dataset.classes)}"
    )


# Apply correct ImageNet labels.

mini_dataset.target_transform = ImageNetLabelMapper(
    mini_dataset.classes,
    wnid_to_idx
)


mini_loader = DataLoader(
    mini_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)


print("\nMiniImageNet loaded successfully")

print("Images:", len(mini_dataset))
print("Classes:", len(mini_dataset.classes))

print("ImageNet label mapping verified.")


# ================================================================
# 7. LOAD PRETRAINED RESNET-18
# ================================================================

print("\nLoading pretrained ImageNet ResNet-18...")


original_model = models.resnet18(
    weights=ResNet18_Weights.DEFAULT
).to(DEVICE)


original_model.eval()


for param in original_model.parameters():
    param.requires_grad = False


# ================================================================
# 8. CREATE STUDENT MODEL
# ================================================================

class ResNet18CIFAR(nn.Module):

    def __init__(self, pretrained_model):

        super().__init__()

        # Copy pretrained ResNet-18

        self.backbone = copy.deepcopy(pretrained_model)

        # Preserve original ImageNet classifier

        self.imagenet_probe = copy.deepcopy(
            pretrained_model.fc
        )

        # Remove FC from backbone

        self.backbone.fc = nn.Identity()

        # New CIFAR-100 classifier

        self.cifar_classifier = nn.Linear(
            512,
            100
        )

        # Freeze all backbone parameters initially.

        for param in self.backbone.parameters():
            param.requires_grad = False

        # Freeze ImageNet evaluation classifier.

        for param in self.imagenet_probe.parameters():
            param.requires_grad = False

        # Enable Layer 1, 2, 3, 4.

        for stage in [

            self.backbone.layer1,
            self.backbone.layer2,
            self.backbone.layer3,
            self.backbone.layer4

        ]:

            for param in stage.parameters():
                param.requires_grad = True

        # CIFAR classifier fully trainable.

        for param in self.cifar_classifier.parameters():
            param.requires_grad = True


    def forward(self, x):

        features = self.backbone(x)

        return self.cifar_classifier(features)


    def imagenet_forward(self, x):

        features = self.backbone(x)

        return self.imagenet_probe(features)


student_model = ResNet18CIFAR(
    original_model
).to(DEVICE)


print("\nStudent model created successfully.")


# ================================================================
# 9. PARAMETER COUNTS
# ================================================================

stages = {

    "Layer 1": student_model.backbone.layer1,

    "Layer 2": student_model.backbone.layer2,

    "Layer 3": student_model.backbone.layer3,

    "Layer 4": student_model.backbone.layer4

}


print("\n" + "=" * 70)
print("PARAMETER DISTRIBUTION")
print("=" * 70)


for name, stage in stages.items():

    count = sum(
        p.numel()
        for p in stage.parameters()
    )

    print(
        f"{name}: {count:,} parameters | "
        f"10% = {int(count * 0.10):,}"
    )


# ================================================================
# 10. RANDOM 10% SELECTION PER STAGE
# ================================================================

current_masks = {}


def generate_stage_masks(model):

    global current_masks

    current_masks = {}

    selected_counts = {}


    for stage_name in [

        "layer1",
        "layer2",
        "layer3",
        "layer4"

    ]:

        stage = getattr(
            model.backbone,
            stage_name
        )

        parameters = list(
            stage.named_parameters()
        )

        total_parameters = sum(
            param.numel()
            for _, param in parameters
        )

        number_selected = max(
            1,
            int(total_parameters * SELECTION_PERCENTAGE)
        )


        # Generate an independent random selection
        # for this stage.

        random_indices = torch.randperm(
            total_parameters,
            device=DEVICE
        )[:number_selected]


        random_indices = torch.sort(
            random_indices
        )[0]


        offset = 0


        for parameter_name, param in parameters:

            number = param.numel()

            local_indices = random_indices[

                (random_indices >= offset) &
                (random_indices < offset + number)

            ] - offset


            mask = torch.zeros(
                number,
                device=DEVICE,
                dtype=torch.bool
            )


            if local_indices.numel() > 0:

                mask[local_indices] = True


            mask = mask.reshape(param.shape)


            full_name = (
                f"backbone.{stage_name}.{parameter_name}"
            )

            current_masks[full_name] = mask


            offset += number


        selected_counts[stage_name] = (
            number_selected,
            total_parameters
        )


    print("\nRANDOM PARAMETER SELECTION")


    for name, (selected, total) in selected_counts.items():

        print(
            f"{name}: {selected:,}/{total:,} "
            f"({selected / total * 100:.2f}%)"
        )


    return current_masks


# ================================================================
# 11. GRADIENT MASK HOOKS
# ================================================================

def register_gradient_hooks(model):

    for name, param in model.named_parameters():

        if not param.requires_grad:
            continue

        if not name.startswith("backbone.layer"):
            continue


        def mask_gradient(
            gradient,
            parameter_name=name
        ):

            mask = current_masks[parameter_name]

            return gradient * mask


        param.register_hook(mask_gradient)


register_gradient_hooks(student_model)


# ================================================================
# 12. OPTIMIZER
# ================================================================

backbone_parameters = []


for stage_name in [

    "layer1",
    "layer2",
    "layer3",
    "layer4"

]:

    stage = getattr(
        student_model.backbone,
        stage_name
    )

    backbone_parameters.extend(
        stage.parameters()
    )


optimizer = optim.SGD(

    [

        {
            "params": backbone_parameters,
            "lr": BACKBONE_LR
        },

        {
            "params": student_model.cifar_classifier.parameters(),
            "lr": CLASSIFIER_LR
        }

    ],

    momentum=0.0,

    weight_decay=0.0

)


scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ================================================================
# 13. FREEZE BATCH NORMALIZATION STATISTICS
# ================================================================

def freeze_batchnorm_statistics(model):

    for module in model.modules():

        if isinstance(module, nn.BatchNorm2d):

            module.eval()


# ================================================================
# 14. CIFAR-100 EVALUATION
# ================================================================

def evaluate_cifar(model):

    model.eval()

    correct = 0

    total = 0

    total_loss = 0.0


    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )


            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                outputs = model(images)

                loss = F.cross_entropy(
                    outputs,
                    labels
                )


            total_loss += (
                loss.item() * labels.size(0)
            )


            predictions = outputs.argmax(dim=1)

            correct += (
                predictions == labels
            ).sum().item()

            total += labels.size(0)


    accuracy = 100.0 * correct / total

    average_loss = total_loss / total

    return accuracy, average_loss


# ================================================================
# 15. ORIGINAL MINIIMAGENET PREDICTIONS
# ================================================================

print("\nComputing original MiniImageNet predictions...")


original_mini_logits = []


with torch.no_grad():

    for images, _ in tqdm(mini_loader):

        images = images.to(
            DEVICE,
            non_blocking=True
        )


        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = original_model(images)


        original_mini_logits.append(
            outputs.float().cpu()
        )


original_mini_logits = torch.cat(
    original_mini_logits,
    dim=0
)


print(
    "Original MiniImageNet logits:",
    original_mini_logits.shape
)


# ================================================================
# 16. MINIIMAGENET EVALUATION
# ================================================================

def evaluate_miniimagenet(model):

    model.eval()

    total_kl = 0.0

    total_agreement = 0

    correct = 0

    total = 0

    start_index = 0


    with torch.no_grad():

        for images, labels in mini_loader:

            images = images.to(
                DEVICE,
                non_blocking=True
            )

            labels = labels.to(
                DEVICE,
                non_blocking=True
            )

            batch_size = images.size(0)


            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                updated_logits = model.imagenet_forward(
                    images
                )


            updated_logits = updated_logits.float()


            original_logits = original_mini_logits[

                start_index:start_index + batch_size

            ].to(DEVICE)


            # Original ImageNet probability distribution

            original_probabilities = F.softmax(
                original_logits,
                dim=1
            )


            # Updated ImageNet log probability distribution

            updated_log_probabilities = F.log_softmax(
                updated_logits,
                dim=1
            )


            # KL divergence:
            # KL(Original || Updated)

            kl = F.kl_div(

                updated_log_probabilities,

                original_probabilities,

                reduction="sum"

            )


            total_kl += kl.item()


            # Prediction agreement with original model

            original_predictions = original_logits.argmax(
                dim=1
            )

            updated_predictions = updated_logits.argmax(
                dim=1
            )


            total_agreement += (

                original_predictions == updated_predictions

            ).sum().item()


            # Actual MiniImageNet classification accuracy

            correct += (

                updated_predictions == labels

            ).sum().item()


            total += batch_size

            start_index += batch_size


    average_kl = total_kl / total

    agreement = (
        100.0 * total_agreement / total
    )

    accuracy = (
        100.0 * correct / total
    )


    return average_kl, agreement, accuracy


# ================================================================
# 17. TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(model):

    model.train()

    freeze_batchnorm_statistics(model)

    total_loss = 0.0

    correct = 0

    total = 0


    for images, labels in tqdm(
        train_loader,
        leave=False
    ):

        images = images.to(
            DEVICE,
            non_blocking=True
        )

        labels = labels.to(
            DEVICE,
            non_blocking=True
        )


        optimizer.zero_grad(set_to_none=True)


        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = model(images)

            loss = F.cross_entropy(
                outputs,
                labels
            )


        scaler.scale(loss).backward()

        scaler.step(optimizer)

        scaler.update()


        total_loss += (
            loss.item() * labels.size(0)
        )


        predictions = outputs.argmax(dim=1)

        correct += (
            predictions == labels
        ).sum().item()

        total += labels.size(0)


    average_loss = total_loss / total

    accuracy = 100.0 * correct / total


    return average_loss, accuracy


# ================================================================
# 18. INITIAL EVALUATION
# ================================================================

print("\n" + "=" * 70)

print("INITIAL MODEL EVALUATION")

print("=" * 70)


initial_cifar_accuracy, initial_cifar_loss = evaluate_cifar(
    student_model
)


initial_mini_kl, initial_mini_agreement, initial_mini_accuracy = (
    evaluate_miniimagenet(student_model)
)


print(
    f"Initial CIFAR Accuracy: "
    f"{initial_cifar_accuracy:.2f}%"
)

print(
    f"Initial CIFAR Loss: "
    f"{initial_cifar_loss:.4f}"
)

print(
    f"Initial MiniImageNet Accuracy: "
    f"{initial_mini_accuracy:.2f}%"
)

print(
    f"Initial MiniImageNet KL: "
    f"{initial_mini_kl:.8f}"
)

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)


# ================================================================
# 19. MAIN ITERATIVE TRAINING
# ================================================================

best_accuracy = initial_cifar_accuracy

best_model_state = copy.deepcopy(
    student_model.state_dict()
)

patience_counter = 0

history = []


print("\n" + "=" * 70)

print("STARTING ITERATIVE TRAINING")

print("=" * 70)


for iteration in range(
    1,
    MAX_ITERATIONS + 1
):

    print("\n")

    print("=" * 70)

    print(
        f"ITERATION {iteration}/{MAX_ITERATIONS}"
    )

    print("=" * 70)


    # ------------------------------------------------------------
    # STEP 1: FRESH RANDOM 10% SELECTION
    # ------------------------------------------------------------

    generate_stage_masks(student_model)


    # ------------------------------------------------------------
    # STEP 2: TRAIN CIFAR-100 FOR 5 EPOCHS
    # ------------------------------------------------------------

    for epoch in range(
        1,
        EPOCHS_PER_ITERATION + 1
    ):

        train_loss, train_accuracy = train_one_epoch(
            student_model
        )


        print(
            f"Iteration {iteration} | "
            f"Epoch {epoch}/{EPOCHS_PER_ITERATION} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}%"
        )


    # ------------------------------------------------------------
    # STEP 3: CIFAR-100 EVALUATION
    # ------------------------------------------------------------

    cifar_accuracy, cifar_loss = evaluate_cifar(
        student_model
    )


    # ------------------------------------------------------------
    # STEP 4: MINIIMAGENET EVALUATION
    # ------------------------------------------------------------

    mini_kl, mini_agreement, mini_accuracy = (
        evaluate_miniimagenet(student_model)
    )


    # ------------------------------------------------------------
    # STEP 5: DISPLAY ALL RESULTS
    # ------------------------------------------------------------

    print("\nITERATION RESULTS")

    print("-" * 50)

    print(
        f"CIFAR-100 Accuracy: {cifar_accuracy:.2f}%"
    )

    print(
        f"CIFAR-100 Loss: {cifar_loss:.4f}"
    )

    print(
        f"MiniImageNet Accuracy: {mini_accuracy:.2f}%"
    )

    print(
        f"MiniImageNet KL Divergence: {mini_kl:.8f}"
    )

    print(
        f"MiniImageNet Prediction Agreement: "
        f"{mini_agreement:.2f}%"
    )


    # ------------------------------------------------------------
    # STEP 6: SAVE HISTORY
    # ------------------------------------------------------------

    history.append({

        "Iteration": iteration,

        "CIFAR_Accuracy": cifar_accuracy,

        "CIFAR_Loss": cifar_loss,

        "MiniImageNet_Accuracy": mini_accuracy,

        "MiniImageNet_KL": mini_kl,

        "MiniImageNet_Agreement": mini_agreement

    })


    # ------------------------------------------------------------
    # STEP 7: SAVE BEST MODEL
    # ------------------------------------------------------------

    if cifar_accuracy > best_accuracy:

        best_accuracy = cifar_accuracy

        best_model_state = copy.deepcopy(
            student_model.state_dict()
        )

        patience_counter = 0


        torch.save(

            best_model_state,

            os.path.join(
                OUTPUT_DIR,
                "best_model.pth"
            )

        )


        print("\nNew best CIFAR-100 model saved.")


    else:

        patience_counter += 1

        print(
            f"No improvement: "
            f"{patience_counter}/{PATIENCE}"
        )


    # ------------------------------------------------------------
    # STEP 8: SAVE HISTORY AFTER EVERY ITERATION
    # ------------------------------------------------------------

    pd.DataFrame(history).to_csv(

        os.path.join(
            OUTPUT_DIR,
            "training_history.csv"
        ),

        index=False

    )


    # ------------------------------------------------------------
    # STEP 9: EARLY STOPPING
    # ------------------------------------------------------------

    if patience_counter >= PATIENCE:

        print("\nEARLY STOPPING TRIGGERED")

        print(
            f"No CIFAR accuracy improvement for "
            f"{PATIENCE} consecutive iterations."
        )

        break


# ================================================================
# 20. LOAD BEST MODEL
# ================================================================

student_model.load_state_dict(
    best_model_state
)


# ================================================================
# 21. FINAL EVALUATION
# ================================================================

print("\n" + "=" * 70)

print("FINAL MODEL EVALUATION")

print("=" * 70)


final_cifar_accuracy, final_cifar_loss = evaluate_cifar(
    student_model
)


final_mini_kl, final_mini_agreement, final_mini_accuracy = (
    evaluate_miniimagenet(student_model)
)


print("\nFINAL RESULTS")

print("-" * 50)

print(
    f"Initial CIFAR Accuracy: "
    f"{initial_cifar_accuracy:.2f}%"
)

print(
    f"Final CIFAR Accuracy: "
    f"{final_cifar_accuracy:.2f}%"
)

print(
    f"Best CIFAR Accuracy: "
    f"{best_accuracy:.2f}%"
)

print(
    f"Initial MiniImageNet Accuracy: "
    f"{initial_mini_accuracy:.2f}%"
)

print(
    f"Final MiniImageNet Accuracy: "
    f"{final_mini_accuracy:.2f}%"
)

print(
    f"Initial MiniImageNet KL: "
    f"{initial_mini_kl:.8f}"
)

print(
    f"Final MiniImageNet KL: "
    f"{final_mini_kl:.8f}"
)

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)

print(
    f"Final MiniImageNet Agreement: "
    f"{final_mini_agreement:.2f}%"
)


# ================================================================
# 22. SAVE FINAL MODEL AND RESULTS
# ================================================================

torch.save(

    student_model.state_dict(),

    os.path.join(
        OUTPUT_DIR,
        "final_model.pth"
    )

)


history_df = pd.DataFrame(history)


history_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "final_training_history.csv"
    ),

    index=False

)


# Save final summary

summary = {

    "Initial_CIFAR_Accuracy": initial_cifar_accuracy,

    "Final_CIFAR_Accuracy": final_cifar_accuracy,

    "Best_CIFAR_Accuracy": best_accuracy,

    "Initial_MiniImageNet_Accuracy": initial_mini_accuracy,

    "Final_MiniImageNet_Accuracy": final_mini_accuracy,

    "Initial_MiniImageNet_KL": initial_mini_kl,

    "Final_MiniImageNet_KL": final_mini_kl,

    "Initial_MiniImageNet_Agreement": initial_mini_agreement,

    "Final_MiniImageNet_Agreement": final_mini_agreement

}


pd.DataFrame(
    [summary]
).to_csv(

    os.path.join(
        OUTPUT_DIR,
        "final_summary.csv"
    ),

    index=False

)


print("\n" + "=" * 70)

print("EXPERIMENT COMPLETED")

print("=" * 70)

print("\nOutput directory:", OUTPUT_DIR)

print("\nFiles saved:")

print("1. best_model.pth")

print("2. final_model.pth")

print("3. training_history.csv")

print("4. final_training_history.csv")

print("5. final_summary.csv")

RESNET-18 RANDOM PARAMETER SELECTION EXPERIMENT
Device: cuda
Image size: 224
Batch size: 64
Selection: 10.0 % per stage
Epochs per iteration: 5
Maximum iterations: 40

CIFAR-100 loaded successfully
Training images: 50000
Testing images: 10000
Classes: 100

ImageNet class mapping loaded
ImageNet classes: 1000

MiniImageNet loaded successfully
Images: 3923
Classes: 1000
ImageNet label mapping verified.

Loading pretrained ImageNet ResNet-18...

Student model created successfully.

PARAMETER DISTRIBUTION
Layer 1: 147,968 parameters | 10% = 14,796
Layer 2: 525,568 parameters | 10% = 52,556
Layer 3: 2,099,712 parameters | 10% = 209,971
Layer 4: 8,393,728 parameters | 10% = 839,372

Computing original MiniImageNet predictions...


  0%|          | 0/62 [00:00<?, ?it/s]

Original MiniImageNet logits: torch.Size([3923, 1000])

INITIAL MODEL EVALUATION
Initial CIFAR Accuracy: 0.79%
Initial CIFAR Loss: 4.8382
Initial MiniImageNet Accuracy: 66.53%
Initial MiniImageNet KL: 0.00000000
Initial MiniImageNet Agreement: 100.00%

STARTING ITERATIVE TRAINING


ITERATION 1/40

RANDOM PARAMETER SELECTION
layer1: 14,796/147,968 (10.00%)
layer2: 52,556/525,568 (10.00%)
layer3: 209,971/2,099,712 (10.00%)
layer4: 839,372/8,393,728 (10.00%)


  0%|          | 0/782 [00:00<?, ?it/s]

Iteration 1 | Epoch 1/5 | Loss: 3.0957 | Train Accuracy: 36.05%


  0%|          | 0/782 [00:00<?, ?it/s]

... (160 outputs left out)

Iteration 17 | Epoch 2/5 | Loss: 0.4393 | Train Accuracy: 86.37%


  0%|          | 0/782 [00:00<?, ?it/s]

Iteration 17 | Epoch 3/5 | Loss: 0.4390 | Train Accuracy: 86.41%


  0%|          | 0/782 [00:00<?, ?it/s]

## CNN

In [ ]:

# ================================================================
# PRETRAINED ViT-B/16
# CIFAR-100 TRAINING + MINIIMAGENET EVALUATION
# Randomly update 1 or 2 Transformer encoder blocks per iteration
# ================================================================

import os
import copy
import json
import random
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from torchvision.models import vit_b_16, ViT_B_16_Weights
from tqdm.auto import tqdm


# ================================================================
# 1. CONFIGURATION
# ================================================================

SEED = 42

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CIFAR_PATH = "/kaggle/input/datasets/melikechan/cifar100/cifar100"

MINI_PATH = (
    "/kaggle/input/datasets/deeptrial/"
    "miniimagenet/ImageNet-Mini/images"
)

IMAGENET_JSON = (
    "/kaggle/input/datasets/deeptrial/"
    "miniimagenet/ImageNet-Mini/imagenet_class_index.json"
)

OUTPUT_DIR = "/kaggle/working/vit_b16_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

IMAGE_SIZE = 224

# ViT-B/16 is relatively large. Start with 16 on a typical GPU.
BATCH_SIZE = 16
NUM_WORKERS = 2

EPOCHS_PER_ITERATION = 5
MAX_ITERATIONS = 40

MIN_SELECTED_BLOCKS = 1
MAX_SELECTED_BLOCKS = 2

PATIENCE = 3

BACKBONE_LR = 1e-5
CLASSIFIER_LR = 1e-3

USE_AMP = DEVICE.type == "cuda"


# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)


print("=" * 70)
print("PRETRAINED ViT-B/16 EXPERIMENT")
print("=" * 70)

print("Device:", DEVICE)
print("Image size:", IMAGE_SIZE, "x", IMAGE_SIZE)
print("Batch size:", BATCH_SIZE)


# ================================================================
# 2. IMAGE TRANSFORMS
# ================================================================

imagenet_mean = [0.485, 0.456, 0.406]
imagenet_std = [0.229, 0.224, 0.225]

cifar_train_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

cifar_test_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])

mini_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE),
        interpolation=transforms.InterpolationMode.BICUBIC
    ),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std)
])


# ================================================================
# 3. LOAD CIFAR-100
# ================================================================

train_dir = os.path.join(CIFAR_PATH, "train")
test_dir = os.path.join(CIFAR_PATH, "test")

if not os.path.isdir(train_dir):
    raise FileNotFoundError(train_dir)

if not os.path.isdir(test_dir):
    raise FileNotFoundError(test_dir)

train_dataset = datasets.ImageFolder(
    train_dir,
    transform=cifar_train_transform
)

test_dataset = datasets.ImageFolder(
    test_dir,
    transform=cifar_test_transform
)

if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("CIFAR train/test class mappings differ.")

if len(train_dataset.classes) != 100:
    raise ValueError(
        f"Expected 100 CIFAR classes, found {len(train_dataset.classes)}"
    )

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

print("\nCIFAR-100 loaded")
print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Classes:", len(train_dataset.classes))


# ================================================================
# 4. LOAD IMAGENET CLASS INDEX
# ================================================================

with open(IMAGENET_JSON, "r") as f:
    imagenet_class_index = json.load(f)

wnid_to_idx = {
    value[0]: int(key)
    for key, value in imagenet_class_index.items()
}

print("\nImageNet class mapping:", len(wnid_to_idx))


# ================================================================
# 5. MINIIMAGENET LABEL MAPPER
# ================================================================

class ImageNetLabelMapper:

    def __init__(self, folder_classes, mapping):

        missing = [
            name for name in folder_classes
            if name not in mapping
        ]

        if missing:
            raise ValueError(
                f"Missing ImageNet mappings: {missing[:10]}"
            )

        self.class_indices = [
            mapping[class_name]
            for class_name in folder_classes
        ]

    def __call__(self, folder_target):
        return self.class_indices[folder_target]


# ================================================================
# 6. LOAD MINIIMAGENET
# ================================================================

if not os.path.isdir(MINI_PATH):
    raise FileNotFoundError(MINI_PATH)

mini_dataset = datasets.ImageFolder(
    MINI_PATH,
    transform=mini_transform
)

if len(mini_dataset.classes) != 1000:
    raise ValueError(
        f"Expected 1000 classes, found {len(mini_dataset.classes)}"
    )

mini_dataset.target_transform = ImageNetLabelMapper(
    mini_dataset.classes,
    wnid_to_idx
)

mini_loader = DataLoader(
    mini_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=True,
    persistent_workers=(NUM_WORKERS > 0)
)

print("\nMiniImageNet loaded")
print("Images:", len(mini_dataset))
print("Classes:", len(mini_dataset.classes))


# Verify input dimensions
sample_images, _ = next(iter(train_loader))
mini_sample_images, _ = next(iter(mini_loader))

print("\nCIFAR batch shape:", sample_images.shape)
print("MiniImageNet batch shape:", mini_sample_images.shape)

assert sample_images.shape[1:] == (3, 224, 224)
assert mini_sample_images.shape[1:] == (3, 224, 224)


# ================================================================
# 7. LOAD PRETRAINED ViT-B/16
# ================================================================

print("\nLoading ImageNet-pretrained ViT-B/16...")

weights = ViT_B_16_Weights.IMAGENET1K_V1

original_model = vit_b_16(weights=weights).to(DEVICE)

original_model.eval()

for param in original_model.parameters():
    param.requires_grad = False

print("Pretrained ViT-B/16 loaded.")

print("Number of Transformer blocks:",
      len(original_model.encoder.layers))

print("Feature dimension: 768")
print("ImageNet output classes: 1000")


# ================================================================
# 8. CREATE STUDENT MODEL
# ================================================================

class ViTStudent(nn.Module):

    def __init__(self, reference_model):
        super().__init__()

        # Copy the complete pretrained ViT
        self.vit = copy.deepcopy(reference_model)

        # Preserve the pretrained ImageNet classification head
        self.imagenet_head = copy.deepcopy(
            reference_model.heads
        )

        # Replace the original head with identity.
        # ViT now returns the 768-dimensional CLS feature.
        self.vit.heads = nn.Identity()

        # New CIFAR-100 classifier
        self.cifar_classifier = nn.Linear(768, 100)

        # Freeze all ViT parameters initially
        for param in self.vit.parameters():
            param.requires_grad = False

        # Enable gradients for Transformer encoder blocks.
        # Gradient hooks will decide which blocks update.
        for block in self.vit.encoder.layers:
            for param in block.parameters():
                param.requires_grad = True

        # ImageNet head remains frozen
        for param in self.imagenet_head.parameters():
            param.requires_grad = False

        # CIFAR classifier is trainable
        for param in self.cifar_classifier.parameters():
            param.requires_grad = True

    def extract_features(self, x):
        return self.vit(x)

    def forward(self, x):
        features = self.extract_features(x)
        return self.cifar_classifier(features)

    def imagenet_forward(self, x):
        features = self.extract_features(x)
        return self.imagenet_head(features)


student_model = ViTStudent(original_model).to(DEVICE)

print("\nStudent ViT created.")


# ================================================================
# 9. RANDOMLY SELECT 1 OR 2 TRANSFORMER BLOCKS
# ================================================================

NUM_BLOCKS = len(student_model.vit.encoder.layers)

selected_blocks = set()


def select_random_blocks():

    global selected_blocks

    number_to_select = random.randint(
        MIN_SELECTED_BLOCKS,
        MAX_SELECTED_BLOCKS
    )

    selected = random.sample(
        range(NUM_BLOCKS),
        number_to_select
    )

    selected_blocks = set(selected)

    print("\nRandomly selected Transformer blocks:")

    for block_index in sorted(selected_blocks):
        print(f"  Encoder block {block_index}")

    print(
        f"Total selected blocks: {len(selected_blocks)}/{NUM_BLOCKS}"
    )

    return selected_blocks


# ================================================================
# 10. GRADIENT MASKING
# ================================================================

def register_block_gradient_hooks(model):

    for block_index, block in enumerate(
        model.vit.encoder.layers
    ):

        for param in block.parameters():

            if not param.requires_grad:
                continue

            def block_gradient_hook(
                gradient,
                index=block_index
            ):

                if index in selected_blocks:
                    return gradient

                return torch.zeros_like(gradient)

            param.register_hook(block_gradient_hook)


register_block_gradient_hooks(student_model)


# ================================================================
# 11. OPTIMIZER
# ================================================================

encoder_parameters = []

for block in student_model.vit.encoder.layers:
    encoder_parameters.extend(block.parameters())

optimizer = optim.AdamW(
    [
        {
            "params": encoder_parameters,
            "lr": BACKBONE_LR
        },
        {
            "params": student_model.cifar_classifier.parameters(),
            "lr": CLASSIFIER_LR
        }
    ],
    weight_decay=0.01
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)


# ================================================================
# 12. CIFAR-100 EVALUATION
# ================================================================

def evaluate_cifar(model):

    model.eval()

    correct = 0
    total = 0
    total_loss = 0.0

    with torch.no_grad():

        for images, labels in test_loader:

            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                outputs = model(images)

                loss = F.cross_entropy(outputs, labels)

            total_loss += loss.item() * labels.size(0)

            predictions = outputs.argmax(dim=1)

            correct += (predictions == labels).sum().item()
            total += labels.size(0)

    accuracy = 100.0 * correct / total
    average_loss = total_loss / total

    return accuracy, average_loss


# ================================================================
# 13. COMPUTE REFERENCE MINIIMAGENET PREDICTIONS
# ================================================================

print("\nComputing pretrained reference predictions...")

original_mini_logits = []

with torch.no_grad():

    for images, _ in tqdm(mini_loader):

        images = images.to(DEVICE, non_blocking=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = original_model(images)

        original_mini_logits.append(
            outputs.float().cpu()
        )

original_mini_logits = torch.cat(
    original_mini_logits,
    dim=0
)

print("Reference logits shape:", original_mini_logits.shape)


# ================================================================
# 14. MINIIMAGENET EVALUATION
# ================================================================

def evaluate_miniimagenet(model):

    model.eval()

    total_kl = 0.0
    total_agreement = 0
    correct = 0
    total = 0

    start_index = 0

    with torch.no_grad():

        for images, labels in mini_loader:

            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)

            batch_size = images.size(0)

            with torch.amp.autocast(
                device_type=DEVICE.type,
                enabled=USE_AMP
            ):

                updated_logits = model.imagenet_forward(images)

            updated_logits = updated_logits.float()

            original_logits = original_mini_logits[
                start_index:start_index + batch_size
            ].to(DEVICE)

            original_probabilities = F.softmax(
                original_logits,
                dim=1
            )

            updated_log_probabilities = F.log_softmax(
                updated_logits,
                dim=1
            )

            kl = F.kl_div(
                updated_log_probabilities,
                original_probabilities,
                reduction="sum"
            )

            total_kl += kl.item()

            original_predictions = original_logits.argmax(dim=1)
            updated_predictions = updated_logits.argmax(dim=1)

            total_agreement += (
                original_predictions == updated_predictions
            ).sum().item()

            correct += (
                updated_predictions == labels
            ).sum().item()

            total += batch_size
            start_index += batch_size

    average_kl = total_kl / total
    agreement = 100.0 * total_agreement / total
    accuracy = 100.0 * correct / total

    return average_kl, agreement, accuracy


# ================================================================
# 15. TRAIN ONE EPOCH
# ================================================================

def train_one_epoch(model):

    model.train()

    total_loss = 0.0
    correct = 0
    total = 0

    for images, labels in tqdm(train_loader, leave=False):

        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(
            device_type=DEVICE.type,
            enabled=USE_AMP
        ):

            outputs = model(images)

            loss = F.cross_entropy(outputs, labels)

        scaler.scale(loss).backward()

        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item() * labels.size(0)

        predictions = outputs.argmax(dim=1)

        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    average_loss = total_loss / total
    accuracy = 100.0 * correct / total

    return average_loss, accuracy


# ================================================================
# 16. INITIAL EVALUATION
# ================================================================

print("\n" + "=" * 70)
print("INITIAL MODEL EVALUATION")
print("=" * 70)

initial_cifar_accuracy, initial_cifar_loss = evaluate_cifar(
    student_model
)

(
    initial_mini_kl,
    initial_mini_agreement,
    initial_mini_accuracy
) = evaluate_miniimagenet(student_model)

print(f"Initial CIFAR Accuracy: {initial_cifar_accuracy:.2f}%")
print(f"Initial CIFAR Loss: {initial_cifar_loss:.4f}")

print(
    f"Initial MiniImageNet Accuracy: "
    f"{initial_mini_accuracy:.2f}%"
)

print(f"Initial MiniImageNet KL: {initial_mini_kl:.8f}")

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)


# ================================================================
# 17. ITERATIVE TRAINING
# ================================================================

best_accuracy = initial_cifar_accuracy

best_model_state = copy.deepcopy(
    student_model.state_dict()
)

patience_counter = 0
history = []

print("\n" + "=" * 70)
print("STARTING ITERATIVE TRAINING")
print("=" * 70)


for iteration in range(1, MAX_ITERATIONS + 1):

    print("\n" + "=" * 70)
    print(f"ITERATION {iteration}/{MAX_ITERATIONS}")
    print("=" * 70)

    # Select a fresh random 1 or 2 encoder blocks
    select_random_blocks()

    # Train CIFAR-100
    for epoch in range(1, EPOCHS_PER_ITERATION + 1):

        train_loss, train_accuracy = train_one_epoch(
            student_model
        )

        print(
            f"Iteration {iteration} | "
            f"Epoch {epoch}/{EPOCHS_PER_ITERATION} | "
            f"Loss: {train_loss:.4f} | "
            f"Train Accuracy: {train_accuracy:.2f}%"
        )

    # CIFAR evaluation
    cifar_accuracy, cifar_loss = evaluate_cifar(
        student_model
    )

    # MiniImageNet evaluation
    mini_kl, mini_agreement, mini_accuracy = (
        evaluate_miniimagenet(student_model)
    )

    print("\nITERATION RESULTS")
    print("-" * 50)

    print(f"CIFAR-100 Accuracy: {cifar_accuracy:.2f}%")
    print(f"CIFAR-100 Loss: {cifar_loss:.4f}")

    print(f"MiniImageNet Accuracy: {mini_accuracy:.2f}%")
    print(f"MiniImageNet KL: {mini_kl:.8f}")

    print(
        f"MiniImageNet Agreement: "
        f"{mini_agreement:.2f}%"
    )

    history.append({
        "Iteration": iteration,
        "Selected_Blocks": str(sorted(selected_blocks)),
        "CIFAR_Accuracy": cifar_accuracy,
        "CIFAR_Loss": cifar_loss,
        "MiniImageNet_Accuracy": mini_accuracy,
        "MiniImageNet_KL": mini_kl,
        "MiniImageNet_Agreement": mini_agreement
    })

    # Save best model
    if cifar_accuracy > best_accuracy:

        best_accuracy = cifar_accuracy

        best_model_state = copy.deepcopy(
            student_model.state_dict()
        )

        patience_counter = 0

        torch.save(
            best_model_state,
            os.path.join(OUTPUT_DIR, "best_model.pth")
        )

        print("\nNew best model saved.")

    else:

        patience_counter += 1

        print(
            f"No improvement: {patience_counter}/{PATIENCE}"
        )

    # Save history after every iteration
    pd.DataFrame(history).to_csv(
        os.path.join(OUTPUT_DIR, "training_history.csv"),
        index=False
    )

    if patience_counter >= PATIENCE:

        print("\nEARLY STOPPING TRIGGERED")
        break


# ================================================================
# 18. LOAD BEST MODEL
# ================================================================

student_model.load_state_dict(best_model_state)


# ================================================================
# 19. FINAL EVALUATION
# ================================================================

print("\n" + "=" * 70)
print("FINAL MODEL EVALUATION")
print("=" * 70)

final_cifar_accuracy, final_cifar_loss = evaluate_cifar(
    student_model
)

(
    final_mini_kl,
    final_mini_agreement,
    final_mini_accuracy
) = evaluate_miniimagenet(student_model)

print("\nFINAL RESULTS")
print("-" * 50)

print(f"Initial CIFAR Accuracy: {initial_cifar_accuracy:.2f}%")
print(f"Final CIFAR Accuracy: {final_cifar_accuracy:.2f}%")
print(f"Best CIFAR Accuracy: {best_accuracy:.2f}%")

print(
    f"Initial MiniImageNet Accuracy: "
    f"{initial_mini_accuracy:.2f}%"
)

print(
    f"Final MiniImageNet Accuracy: "
    f"{final_mini_accuracy:.2f}%"
)

print(f"Initial MiniImageNet KL: {initial_mini_kl:.8f}")
print(f"Final MiniImageNet KL: {final_mini_kl:.8f}")

print(
    f"Initial MiniImageNet Agreement: "
    f"{initial_mini_agreement:.2f}%"
)

print(
    f"Final MiniImageNet Agreement: "
    f"{final_mini_agreement:.2f}%"
)


# ================================================================
# 20. SAVE MODEL AND RESULTS
# ================================================================

torch.save(
    student_model.state_dict(),
    os.path.join(OUTPUT_DIR, "final_model.pth")
)

history_df = pd.DataFrame(history)

history_df.to_csv(
    os.path.join(OUTPUT_DIR, "final_training_history.csv"),
    index=False
)

summary = {
    "Architecture": "ViT-B/16",
    "Pretrained_Weights": "ImageNet-1K",
    "Image_Size": IMAGE_SIZE,
    "Initial_CIFAR_Accuracy": initial_cifar_accuracy,
    "Final_CIFAR_Accuracy": final_cifar_accuracy,
    "Best_CIFAR_Accuracy": best_accuracy,
    "Initial_MiniImageNet_Accuracy": initial_mini_accuracy,
    "Final_MiniImageNet_Accuracy": final_mini_accuracy,
    "Initial_MiniImageNet_KL": initial_mini_kl,
    "Final_MiniImageNet_KL": final_mini_kl,
    "Initial_MiniImageNet_Agreement": initial_mini_agreement,
    "Final_MiniImageNet_Agreement": final_mini_agreement
}

pd.DataFrame([summary]).to_csv(
    os.path.join(OUTPUT_DIR, "final_summary.csv"),
    index=False
)

print("\n" + "=" * 70)
print("EXPERIMENT COMPLETED")
print("=" * 70)

print("Output directory:", OUTPUT_DIR)

print("\nSaved files:")
print("1. best_model.pth")
print("2. final_model.pth")
print("3. training_history.csv")
print("4. final_training_history.csv")
print("5. final_summary.csv")


PRETRAINED ViT-B/16 EXPERIMENT
Device: cuda
Image size: 224 x 224
Batch size: 16

CIFAR-100 loaded
Train images: 50000
Test images: 10000
Classes: 100

ImageNet class mapping: 1000

MiniImageNet loaded
Images: 3923
Classes: 1000

CIFAR batch shape: torch.Size([16, 3, 224, 224])
MiniImageNet batch shape: torch.Size([16, 3, 224, 224])

Loading ImageNet-pretrained ViT-B/16...
Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:01<00:00, 187MB/s]  


Pretrained ViT-B/16 loaded.
Number of Transformer blocks: 12
Feature dimension: 768
ImageNet output classes: 1000

Student ViT created.

Computing pretrained reference predictions...


  0%|          | 0/246 [00:00<?, ?it/s]

Reference logits shape: torch.Size([3923, 1000])

INITIAL MODEL EVALUATION
Initial CIFAR Accuracy: 0.92%
Initial CIFAR Loss: 4.6947
Initial MiniImageNet Accuracy: 79.56%
Initial MiniImageNet KL: -0.00000000
Initial MiniImageNet Agreement: 100.00%

STARTING ITERATIVE TRAINING

ITERATION 1/40

Randomly selected Transformer blocks:
  Encoder block 0
Total selected blocks: 1/12


  0%|          | 0/3125 [00:00<?, ?it/s]

... (10 outputs left out)

Iteration 2 | Epoch 1/5 | Loss: 0.1854 | Train Accuracy: 93.87%


  0%|          | 0/3125 [00:00<?, ?it/s]

Iteration 2 | Epoch 2/5 | Loss: 0.1144 | Train Accuracy: 96.34%


  0%|          | 0/3125 [00:00<?, ?it/s]

## truncate

In [20]:

import os
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

# -------------------- CONFIGURATION --------------------
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CIFAR_PATH = "/kaggle/input/datasets/melikechan/cifar100/cifar100"
MINI_PATH = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/images"
IMAGENET_JSON = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/imagenet_class_index.json"
OUTPUT_DIR = "/kaggle/working/resnet18_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 64
IMAGE_SIZE = 224
EPOCHS_PER_ITERATION = 3
MAX_ITERATIONS = 20
SELECTION_PERCENTAGE = 0.10
OLD_WEIGHT_FACTOR = 0.90
NEW_WEIGHT_FACTOR = 0.10
PATIENCE = 3
BACKBONE_LR = 0.0001
CLASSIFIER_LR = 0.01
NUM_WORKERS = 2
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------- DATA --------------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(CIFAR_PATH, "train"), transform=train_transform)
test_dataset = datasets.ImageFolder(os.path.join(CIFAR_PATH, "test"), transform=test_transform)
if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("CIFAR train/test class mappings differ.")
if len(train_dataset.classes) != 100:
    raise ValueError("Expected CIFAR-100 classes.")

loader_args = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

with open(IMAGENET_JSON, "r") as f:
    imagenet_class_index = json.load(f)
wnid_to_idx = {value[0]: int(key) for key, value in imagenet_class_index.items()}

class ImageNetLabelMapper:
    def __init__(self, folder_classes, mapping):
        missing = [name for name in folder_classes if name not in mapping]
        if missing:
            raise ValueError(f"Missing ImageNet mappings: {missing[:10]}")
        self.indices = [mapping[name] for name in folder_classes]

    def __call__(self, target):
        return self.indices[target]

mini_dataset = datasets.ImageFolder(MINI_PATH, transform=test_transform)
if len(mini_dataset.classes) != 1000:
    raise ValueError(f"Expected 1000 MiniImageNet classes, found {len(mini_dataset.classes)}")
mini_dataset.target_transform = ImageNetLabelMapper(mini_dataset.classes, wnid_to_idx)
mini_loader = DataLoader(mini_dataset, shuffle=False, **loader_args)

# -------------------- MODELS --------------------
original_model = models.resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE)
original_model.eval()
for p in original_model.parameters():
    p.requires_grad = False

class ResNet18CIFAR(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = copy.deepcopy(pretrained)
        self.imagenet_probe = copy.deepcopy(pretrained.fc)
        self.backbone.fc = nn.Identity()
        self.cifar_classifier = nn.Linear(512, 100)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for p in self.imagenet_probe.parameters():
            p.requires_grad = False
        for stage in [self.backbone.layer1, self.backbone.layer2,
                      self.backbone.layer3, self.backbone.layer4]:
            for p in stage.parameters():
                p.requires_grad = True
        for p in self.cifar_classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.cifar_classifier(self.backbone(x))

    def imagenet_forward(self, x):
        return self.imagenet_probe(self.backbone(x))

student_model = ResNet18CIFAR(original_model).to(DEVICE)

# -------------------- PARAMETER SELECTION + THRESHOLD --------------------
STAGE_NAMES = ["layer1", "layer2", "layer3", "layer4"]
current_masks = {}
current_update_masks = {}

def generate_stage_masks(model):
    global current_masks
    current_masks = {}
    for stage_name in STAGE_NAMES:
        stage = getattr(model.backbone, stage_name)
        params = list(stage.named_parameters())
        total = sum(p.numel() for _, p in params)
        selected_count = max(1, int(total * SELECTION_PERCENTAGE))
        indices = torch.randperm(total, device=DEVICE)[:selected_count].sort()[0]
        offset = 0
        for param_name, param in params:
            count = param.numel()
            local = indices[(indices >= offset) & (indices < offset + count)] - offset
            mask = torch.zeros(count, device=DEVICE, dtype=torch.bool)
            if local.numel():
                mask[local] = True
            current_masks[f"backbone.{stage_name}.{param_name}"] = mask.view_as(param)
            offset += count
    return current_masks

def build_threshold_update_masks(model):
    global current_update_masks
    current_update_masks = {}
    for stage_name in STAGE_NAMES:
        stage = getattr(model.backbone, stage_name)
        threshold = torch.cat([p.detach().abs().flatten() for p in stage.parameters()]).mean()
        for param_name, param in stage.named_parameters():
            name = f"backbone.{stage_name}.{param_name}"
            current_update_masks[name] = current_masks[name] & (param.detach().abs() < threshold)
    return current_update_masks

def register_gradient_hooks(model):
    for name, param in model.named_parameters():
        if param.requires_grad and name.startswith("backbone.layer"):
            param.register_hook(lambda grad, n=name: grad * current_update_masks[n])

register_gradient_hooks(student_model)

# -------------------- OPTIMIZER --------------------
backbone_params = []
for stage_name in STAGE_NAMES:
    backbone_params.extend(getattr(student_model.backbone, stage_name).parameters())
optimizer = optim.SGD([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": student_model.cifar_classifier.parameters(), "lr": CLASSIFIER_LR}
], momentum=0.0, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def freeze_batchnorm_statistics(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

# -------------------- EVALUATION --------------------
def evaluate_cifar(model):
    model.eval()
    correct = total = 0
    total_loss = 0.0
    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                outputs = model(images)
                loss = F.cross_entropy(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total, total_loss / total

# Store original ImageNet predictions once for KL and agreement comparisons.
original_mini_logits = []
with torch.inference_mode():
    for images, _ in mini_loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = original_model(images)
        original_mini_logits.append(logits.float().cpu())
original_mini_logits = torch.cat(original_mini_logits, dim=0)

def evaluate_miniimagenet(model):
    model.eval()
    total_kl = 0.0
    total_agreement = correct = total = start = 0
    with torch.inference_mode():
        for images, labels in mini_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            batch = images.size(0)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                updated_logits = model.imagenet_forward(images).float()
            original_logits = original_mini_logits[start:start + batch].to(DEVICE)
            original_probs = F.softmax(original_logits, dim=1)
            updated_log_probs = F.log_softmax(updated_logits, dim=1)
            total_kl += F.kl_div(updated_log_probs, original_probs, reduction="sum").item()
            original_pred = original_logits.argmax(1)
            updated_pred = updated_logits.argmax(1)
            total_agreement += (original_pred == updated_pred).sum().item()
            correct += (updated_pred == labels).sum().item()
            total += batch
            start += batch
    return total_kl / total, 100.0 * total_agreement / total, 100.0 * correct / total

# -------------------- TRAINING --------------------
def train_one_epoch(model):
    model.train()
    freeze_batchnorm_statistics(model)
    total_loss = correct = total = 0
    params = dict(model.named_parameters())
    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # Save only eligible selected values, not whole tensors.
        old_values = {
            name: params[name].detach()[mask].clone()
            for name, mask in current_update_masks.items() if mask.any()
        }
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Apply 90% old weight + 10% SGD candidate weight.
        with torch.no_grad():
            for name, old in old_values.items():
                mask = current_update_masks[name]
                param = params[name]
                candidate = param[mask].clone()
                param[mask] = OLD_WEIGHT_FACTOR * old + NEW_WEIGHT_FACTOR * candidate

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

# -------------------- MAIN --------------------
initial_cifar_accuracy, initial_cifar_loss = evaluate_cifar(student_model)
initial_mini_kl, initial_mini_agreement, initial_mini_accuracy = evaluate_miniimagenet(student_model)
print(f"Device: {DEVICE} | Initial CIFAR: {initial_cifar_accuracy:.2f}% | Initial MiniImageNet: {initial_mini_accuracy:.2f}%")

best_accuracy = initial_cifar_accuracy
best_model_state = copy.deepcopy(student_model.state_dict())
patience_counter = 0
history = []

for iteration in range(1, MAX_ITERATIONS + 1):
    # Select 10% once and calculate each stage threshold once per iteration.
    generate_stage_masks(student_model)
    build_threshold_update_masks(student_model)

    for epoch in range(1, EPOCHS_PER_ITERATION + 1):
        train_loss, train_accuracy = train_one_epoch(student_model)
        print(f"Iteration {iteration} | Epoch {epoch}/{EPOCHS_PER_ITERATION} | Loss {train_loss:.4f} | Train Acc {train_accuracy:.2f}%")

    cifar_accuracy, cifar_loss = evaluate_cifar(student_model)

    # Evaluate MiniImageNet at every iteration.
    mini_kl, mini_agreement, mini_accuracy = evaluate_miniimagenet(student_model)

    print(
        f"Iteration {iteration}/{MAX_ITERATIONS} | "
        f"CIFAR Acc: {cifar_accuracy:.2f}% | "
        f"MiniImageNet Acc: {mini_accuracy:.2f}% | "
        f"KL: {mini_kl:.6f} | "
        f"Agreement: {mini_agreement:.2f}%"
    )

    history.append({
        "Iteration": iteration,
        "CIFAR_Accuracy": cifar_accuracy,
        "CIFAR_Loss": cifar_loss,
        "MiniImageNet_Accuracy": mini_accuracy,
        "MiniImageNet_KL": mini_kl,
        "MiniImageNet_Agreement": mini_agreement
    })

    if cifar_accuracy > best_accuracy:
        best_accuracy = cifar_accuracy
        best_model_state = copy.deepcopy(student_model.state_dict())
        patience_counter = 0
        torch.save(best_model_state, os.path.join(OUTPUT_DIR, "best_model.pth"))
    else:
        patience_counter += 1

    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

# -------------------- FINAL EVALUATION + SAVING --------------------
student_model.load_state_dict(best_model_state)
final_cifar_accuracy, final_cifar_loss = evaluate_cifar(student_model)
final_mini_kl, final_mini_agreement, final_mini_accuracy = evaluate_miniimagenet(student_model)
print(f"FINAL | CIFAR: {final_cifar_accuracy:.2f}% (best {best_accuracy:.2f}%) | MiniImageNet: {final_mini_accuracy:.2f}% | KL: {final_mini_kl:.6f} | Agreement: {final_mini_agreement:.2f}%")

torch.save(student_model.state_dict(), os.path.join(OUTPUT_DIR, "final_model.pth"))
pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "final_training_history.csv"), index=False)
summary = {
    "Initial_CIFAR_Accuracy": initial_cifar_accuracy,
    "Final_CIFAR_Accuracy": final_cifar_accuracy,
    "Best_CIFAR_Accuracy": best_accuracy,
    "Initial_MiniImageNet_Accuracy": initial_mini_accuracy,
    "Final_MiniImageNet_Accuracy": final_mini_accuracy,
    "Initial_MiniImageNet_KL": initial_mini_kl,
    "Final_MiniImageNet_KL": final_mini_kl,
    "Initial_MiniImageNet_Agreement": initial_mini_agreement,
    "Final_MiniImageNet_Agreement": final_mini_agreement
}
pd.DataFrame([summary]).to_csv(os.path.join(OUTPUT_DIR, "final_summary.csv"), index=False)
print(f"Completed. Results saved in: {OUTPUT_DIR}")


Device: cuda | Initial CIFAR: 0.79% | Initial MiniImageNet: 66.53%
Iteration 1 | Epoch 1/3 | Loss 3.2082 | Train Acc 34.76%
Iteration 1 | Epoch 2/3 | Loss 1.9931 | Train Acc 56.31%
Iteration 1 | Epoch 3/3 | Loss 1.6295 | Train Acc 61.10%
Iteration 1/20 | CIFAR Acc: 61.27% | MiniImageNet Acc: 66.53% | KL: 0.001332 | Agreement: 98.45%
Iteration 2 | Epoch 1/3 | Loss 1.4484 | Train Acc 63.46%
Iteration 2 | Epoch 2/3 | Loss 1.3386 | Train Acc 65.27%
Iteration 2 | Epoch 3/3 | Loss 1.2587 | Train Acc 66.72%
Iteration 2/20 | CIFAR Acc: 65.05% | MiniImageNet Acc: 66.61% | KL: 0.004520 | Agreement: 97.32%
Iteration 3 | Epoch 1/3 | Loss 1.1994 | Train Acc 67.76%
Iteration 3 | Epoch 2/3 | Loss 1.1542 | Train Acc 68.53%
Iteration 3 | Epoch 3/3 | Loss 1.1162 | Train Acc 69.23%
Iteration 3/20 | CIFAR Acc: 67.14% | MiniImageNet Acc: 66.63% | KL: 0.008403 | Agreement: 96.35%
Iteration 4 | Epoch 1/3 | Loss 1.0848 | Train Acc 69.92%
Iteration 4 | Epoch 2/3 | Loss 1.0565 | Train Acc 70.62%
Iteration 4 | E

In [ ]:

import os
import copy
import json
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

# -------------------- CONFIGURATION --------------------
SEED = 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
CIFAR_PATH = "/kaggle/input/datasets/melikechan/cifar100/cifar100"
MINI_PATH = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/images"
IMAGENET_JSON = "/kaggle/input/datasets/deeptrial/miniimagenet/ImageNet-Mini/imagenet_class_index.json"
OUTPUT_DIR = "/kaggle/working/resnet18_experiment"
os.makedirs(OUTPUT_DIR, exist_ok=True)

BATCH_SIZE = 64
IMAGE_SIZE = 224
EPOCHS_PER_ITERATION = 5
MAX_ITERATIONS = 20
SELECTION_PERCENTAGE = 0.10
OLD_WEIGHT_FACTOR = 0.90
NEW_WEIGHT_FACTOR = 0.10
PATIENCE = 3
BACKBONE_LR = 0.0001
CLASSIFIER_LR = 0.01
NUM_WORKERS = 2
USE_AMP = DEVICE.type == "cuda"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# -------------------- DATA --------------------
train_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

train_dataset = datasets.ImageFolder(os.path.join(CIFAR_PATH, "train"), transform=train_transform)
test_dataset = datasets.ImageFolder(os.path.join(CIFAR_PATH, "test"), transform=test_transform)
if train_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("CIFAR train/test class mappings differ.")
if len(train_dataset.classes) != 100:
    raise ValueError("Expected CIFAR-100 classes.")

loader_args = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": DEVICE.type == "cuda",
    "persistent_workers": NUM_WORKERS > 0
}
train_loader = DataLoader(train_dataset, shuffle=True, **loader_args)
test_loader = DataLoader(test_dataset, shuffle=False, **loader_args)

with open(IMAGENET_JSON, "r") as f:
    imagenet_class_index = json.load(f)
wnid_to_idx = {value[0]: int(key) for key, value in imagenet_class_index.items()}

class ImageNetLabelMapper:
    def __init__(self, folder_classes, mapping):
        missing = [name for name in folder_classes if name not in mapping]
        if missing:
            raise ValueError(f"Missing ImageNet mappings: {missing[:10]}")
        self.indices = [mapping[name] for name in folder_classes]

    def __call__(self, target):
        return self.indices[target]

mini_dataset = datasets.ImageFolder(MINI_PATH, transform=test_transform)
if len(mini_dataset.classes) != 1000:
    raise ValueError(f"Expected 1000 MiniImageNet classes, found {len(mini_dataset.classes)}")
mini_dataset.target_transform = ImageNetLabelMapper(mini_dataset.classes, wnid_to_idx)
mini_loader = DataLoader(mini_dataset, shuffle=False, **loader_args)

# -------------------- MODELS --------------------
original_model = models.resnet18(weights=ResNet18_Weights.DEFAULT).to(DEVICE)
original_model.eval()
for p in original_model.parameters():
    p.requires_grad = False

class ResNet18CIFAR(nn.Module):
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = copy.deepcopy(pretrained)
        self.imagenet_probe = copy.deepcopy(pretrained.fc)
        self.backbone.fc = nn.Identity()
        self.cifar_classifier = nn.Linear(512, 100)
        for p in self.backbone.parameters():
            p.requires_grad = False
        for p in self.imagenet_probe.parameters():
            p.requires_grad = False
        for stage in [self.backbone.layer1, self.backbone.layer2,
                      self.backbone.layer3, self.backbone.layer4]:
            for p in stage.parameters():
                p.requires_grad = True
        for p in self.cifar_classifier.parameters():
            p.requires_grad = True

    def forward(self, x):
        return self.cifar_classifier(self.backbone(x))

    def imagenet_forward(self, x):
        return self.imagenet_probe(self.backbone(x))

student_model = ResNet18CIFAR(original_model).to(DEVICE)

# -------------------- PARAMETER SELECTION + THRESHOLD --------------------
STAGE_NAMES = ["layer1", "layer2", "layer3", "layer4"]
current_masks = {}
current_update_masks = {}

def generate_stage_masks(model):
    global current_masks
    current_masks = {}
    for stage_name in STAGE_NAMES:
        stage = getattr(model.backbone, stage_name)
        params = list(stage.named_parameters())
        total = sum(p.numel() for _, p in params)
        selected_count = max(1, int(total * SELECTION_PERCENTAGE))
        indices = torch.randperm(total, device=DEVICE)[:selected_count].sort()[0]
        offset = 0
        for param_name, param in params:
            count = param.numel()
            local = indices[(indices >= offset) & (indices < offset + count)] - offset
            mask = torch.zeros(count, device=DEVICE, dtype=torch.bool)
            if local.numel():
                mask[local] = True
            current_masks[f"backbone.{stage_name}.{param_name}"] = mask.view_as(param)
            offset += count
    return current_masks

def build_threshold_update_masks(model):
    global current_update_masks
    current_update_masks = {}
    for stage_name in STAGE_NAMES:
        stage = getattr(model.backbone, stage_name)
        threshold = torch.cat([p.detach().abs().flatten() for p in stage.parameters()]).mean()
        for param_name, param in stage.named_parameters():
            name = f"backbone.{stage_name}.{param_name}"
            current_update_masks[name] = current_masks[name] & (param.detach().abs() < threshold)
    return current_update_masks

def register_gradient_hooks(model):
    for name, param in model.named_parameters():
        if param.requires_grad and name.startswith("backbone.layer"):
            param.register_hook(lambda grad, n=name: grad * current_update_masks[n])

register_gradient_hooks(student_model)

# -------------------- OPTIMIZER --------------------
backbone_params = []
for stage_name in STAGE_NAMES:
    backbone_params.extend(getattr(student_model.backbone, stage_name).parameters())
optimizer = optim.SGD([
    {"params": backbone_params, "lr": BACKBONE_LR},
    {"params": student_model.cifar_classifier.parameters(), "lr": CLASSIFIER_LR}
], momentum=0.0, weight_decay=0.0)
scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)

def freeze_batchnorm_statistics(model):
    for module in model.modules():
        if isinstance(module, nn.BatchNorm2d):
            module.eval()

# -------------------- EVALUATION --------------------
def evaluate_cifar(model):
    model.eval()
    correct = total = 0
    total_loss = 0.0
    with torch.inference_mode():
        for images, labels in test_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                outputs = model(images)
                loss = F.cross_entropy(outputs, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total, total_loss / total

# Store original ImageNet predictions once for KL and agreement comparisons.
original_mini_logits = []
with torch.inference_mode():
    for images, _ in mini_loader:
        images = images.to(DEVICE, non_blocking=True)
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            logits = original_model(images)
        original_mini_logits.append(logits.float().cpu())
original_mini_logits = torch.cat(original_mini_logits, dim=0)

def evaluate_miniimagenet(model):
    model.eval()
    total_kl = 0.0
    total_agreement = correct = total = start = 0
    with torch.inference_mode():
        for images, labels in mini_loader:
            images = images.to(DEVICE, non_blocking=True)
            labels = labels.to(DEVICE, non_blocking=True)
            batch = images.size(0)
            with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
                updated_logits = model.imagenet_forward(images).float()
            original_logits = original_mini_logits[start:start + batch].to(DEVICE)
            original_probs = F.softmax(original_logits, dim=1)
            updated_log_probs = F.log_softmax(updated_logits, dim=1)
            total_kl += F.kl_div(updated_log_probs, original_probs, reduction="sum").item()
            original_pred = original_logits.argmax(1)
            updated_pred = updated_logits.argmax(1)
            total_agreement += (original_pred == updated_pred).sum().item()
            correct += (updated_pred == labels).sum().item()
            total += batch
            start += batch
    return total_kl / total, 100.0 * total_agreement / total, 100.0 * correct / total

# -------------------- TRAINING --------------------
def train_one_epoch(model):
    model.train()
    freeze_batchnorm_statistics(model)
    total_loss = correct = total = 0
    params = dict(model.named_parameters())
    for images, labels in train_loader:
        images = images.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        # Save only eligible selected values, not whole tensors.
        old_values = {
            name: params[name].detach()[mask].clone()
            for name, mask in current_update_masks.items() if mask.any()
        }
        with torch.amp.autocast(device_type=DEVICE.type, enabled=USE_AMP):
            outputs = model(images)
            loss = F.cross_entropy(outputs, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        # Apply 90% old weight + 10% SGD candidate weight.
        with torch.no_grad():
            for name, old in old_values.items():
                mask = current_update_masks[name]
                param = params[name]
                candidate = param[mask].clone()
                param[mask] = OLD_WEIGHT_FACTOR * old + NEW_WEIGHT_FACTOR * candidate

        total_loss += loss.item() * labels.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return total_loss / total, 100.0 * correct / total

# -------------------- MAIN --------------------
initial_cifar_accuracy, initial_cifar_loss = evaluate_cifar(student_model)
initial_mini_kl, initial_mini_agreement, initial_mini_accuracy = evaluate_miniimagenet(student_model)
print(f"Device: {DEVICE} | Initial CIFAR: {initial_cifar_accuracy:.2f}% | Initial MiniImageNet: {initial_mini_accuracy:.2f}%")

best_accuracy = initial_cifar_accuracy
best_model_state = copy.deepcopy(student_model.state_dict())
patience_counter = 0
history = []

for iteration in range(1, MAX_ITERATIONS + 1):
    # Select 10% once and calculate each stage threshold once per iteration.
    generate_stage_masks(student_model)
    build_threshold_update_masks(student_model)

    for epoch in range(1, EPOCHS_PER_ITERATION + 1):
        train_loss, train_accuracy = train_one_epoch(student_model)
        print(f"Iteration {iteration} | Epoch {epoch}/{EPOCHS_PER_ITERATION} | Loss {train_loss:.4f} | Train Acc {train_accuracy:.2f}%")

    cifar_accuracy, cifar_loss = evaluate_cifar(student_model)

    # Evaluate MiniImageNet at every iteration.
    mini_kl, mini_agreement, mini_accuracy = evaluate_miniimagenet(student_model)

    print(
        f"Iteration {iteration}/{MAX_ITERATIONS} | "
        f"CIFAR Acc: {cifar_accuracy:.2f}% | "
        f"MiniImageNet Acc: {mini_accuracy:.2f}% | "
        f"KL: {mini_kl:.6f} | "
        f"Agreement: {mini_agreement:.2f}%"
    )

    history.append({
        "Iteration": iteration,
        "CIFAR_Accuracy": cifar_accuracy,
        "CIFAR_Loss": cifar_loss,
        "MiniImageNet_Accuracy": mini_accuracy,
        "MiniImageNet_KL": mini_kl,
        "MiniImageNet_Agreement": mini_agreement
    })

    if cifar_accuracy > best_accuracy:
        best_accuracy = cifar_accuracy
        best_model_state = copy.deepcopy(student_model.state_dict())
        patience_counter = 0
        torch.save(best_model_state, os.path.join(OUTPUT_DIR, "best_model.pth"))
    else:
        patience_counter += 1

    pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "training_history.csv"), index=False)

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

# -------------------- FINAL EVALUATION + SAVING --------------------
student_model.load_state_dict(best_model_state)
final_cifar_accuracy, final_cifar_loss = evaluate_cifar(student_model)
final_mini_kl, final_mini_agreement, final_mini_accuracy = evaluate_miniimagenet(student_model)
print(f"FINAL | CIFAR: {final_cifar_accuracy:.2f}% (best {best_accuracy:.2f}%) | MiniImageNet: {final_mini_accuracy:.2f}% | KL: {final_mini_kl:.6f} | Agreement: {final_mini_agreement:.2f}%")

torch.save(student_model.state_dict(), os.path.join(OUTPUT_DIR, "final_model.pth"))
pd.DataFrame(history).to_csv(os.path.join(OUTPUT_DIR, "final_training_history.csv"), index=False)
summary = {
    "Initial_CIFAR_Accuracy": initial_cifar_accuracy,
    "Final_CIFAR_Accuracy": final_cifar_accuracy,
    "Best_CIFAR_Accuracy": best_accuracy,
    "Initial_MiniImageNet_Accuracy": initial_mini_accuracy,
    "Final_MiniImageNet_Accuracy": final_mini_accuracy,
    "Initial_MiniImageNet_KL": initial_mini_kl,
    "Final_MiniImageNet_KL": final_mini_kl,
    "Initial_MiniImageNet_Agreement": initial_mini_agreement,
    "Final_MiniImageNet_Agreement": final_mini_agreement
}
pd.DataFrame([summary]).to_csv(os.path.join(OUTPUT_DIR, "final_summary.csv"), index=False)
print(f"Completed. Results saved in: {OUTPUT_DIR}")


Device: cuda | Initial CIFAR: 0.79% | Initial MiniImageNet: 66.53%
Iteration 1 | Epoch 1/5 | Loss 3.2082 | Train Acc 34.76%
Iteration 1 | Epoch 2/5 | Loss 1.9931 | Train Acc 56.31%
Iteration 1 | Epoch 3/5 | Loss 1.6295 | Train Acc 61.10%
Iteration 1 | Epoch 4/5 | Loss 1.4484 | Train Acc 63.47%
Iteration 1 | Epoch 5/5 | Loss 1.3386 | Train Acc 65.25%
Iteration 1/20 | CIFAR Acc: 64.09% | MiniImageNet Acc: 66.61% | KL: 0.003338 | Agreement: 97.68%
Iteration 2 | Epoch 1/5 | Loss 1.2588 | Train Acc 66.71%
Iteration 2 | Epoch 2/5 | Loss 1.1995 | Train Acc 67.77%
Iteration 2 | Epoch 3/5 | Loss 1.1545 | Train Acc 68.52%
Iteration 2 | Epoch 4/5 | Loss 1.1165 | Train Acc 69.24%
Iteration 2 | Epoch 5/5 | Loss 1.0851 | Train Acc 69.89%
Iteration 2/20 | CIFAR Acc: 67.48% | MiniImageNet Acc: 66.68% | KL: 0.009530 | Agreement: 96.00%
Iteration 3 | Epoch 1/5 | Loss 1.0568 | Train Acc 70.63%
Iteration 3 | Epoch 2/5 | Loss 1.0313 | Train Acc 71.00%
Iteration 3 | Epoch 3/5 | Loss 1.0113 | Train Acc 71.28